# TAC calibration — 02 track access charges

Single source of truth for every calibrated TAC value, and the generator for
`TAC_CALIBRATION.md`. Run `01_source_extraction.ipynb` first.

Scope is the **minimum access package only**. Energy, station and facility
charges belong to their own domains and are excluded here even where a national
tariff bundles them together.

Outputs (all generated, none hand-edited): `data/tac_components.csv`,
`data/tac_night_mode.csv`, `data/tac_peak_bands.csv`, `data/passage_charges.csv`,
`data/passage_geometries.geojson`, and `TAC_CALIBRATION.md`.

In [ ]:
# TAC calibration — track access charges for the minimum access package
#
# Single source of truth for every calibrated TAC value. The CSVs under
# `data/` and the document `TAC_CALIBRATION.md` are generated artifacts:
# re-run this notebook (after 01) to regenerate them, never hand-edit.
#
# Scope is deliberately narrow: the minimum access package (MAP) only.
# Energy, station and facility charges are calibrated in their own domains
# and must not appear here — see the scope section of the generated
# document for what is excluded and why.

import csv
import json
from dataclasses import dataclass, asdict, field
from pathlib import Path


def _resolve_data_dir() -> Path:
    here = Path.cwd()
    for cand in (here / "data", here / "backend/models/infrastructure/tac/calib/data"):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
print(f"data directory: {DATA_DIR}")

# Date this calibration was last reviewed end to end. Held here rather than
# taken from the register so 02 stands alone: the notebooks share files, not
# Python state, and either can be re-run without the other in memory.
CALIBRATION_REVIEWED = "2026-08-11"

# --- provenance vocabulary -------------------------------------------------
# How much evidence stands behind a value. Not a quality judgement — a
# well-argued ASSUMED and a mis-transcribed SOURCED are both possible; the
# status says which kind of thing the reader is looking at.
SOURCED = "sourced"  # named document, named locator
DERIVED = "derived"  # arithmetic on other values, formula in the note
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
MISSING = "missing"  # nothing read
NO_RAILWAY = "no_railway"  # positively documented as having no network

# --- price basis and conversion -------------------------------------------
# The evaluation year of the target network. Every monetary value is carried
# here before it reaches the database.
TARGET_YEAR = 2032

# Passenger MAP charges per train-km have risen ~3% a year across Europe for
# a decade: EUR 4.13 (2015) to 4.63 (2019) per IRG-MM-9, and a continuous
# ~3%/yr over 2020-2024 per IRG-MM-14. Two independent five-year windows
# agreeing makes this a trend rather than an inflation-spike artefact. It
# sits above HICP (~2%), i.e. track access rises ~1%/yr in real terms —
# structural, from full-cost recovery under Directive 2012/34 Art.31-32
# against a growing renewal burden.
#
# Germany runs hotter (SPFV +19.9% over 2020-2024) but is deliberately NOT
# extrapolated: that gap came from the Trassenpreisbremse pushing the
# regional shortfall onto long-distance, and the ECJ struck the provision
# down in March 2026 (C-770/24), so the mechanism is unwinding.
TAC_ESCALATION_PER_YEAR = 0.03
TAC_ESCALATION_LOW = 0.020  # HICP only — assumes the real-terms rise stops
TAC_ESCALATION_HIGH = 0.035  # the 2015-2024 trend steepens with the backlog

# Per-country deviations from the European rate, with a mandatory reason.
# The default is a European average and an average is the wrong instrument
# where a national tariff is demonstrably not moving with it — but a
# deviation is a claim about one country's next decade, so it is stated
# here rather than buried in a note.
ESCALATION_OVERRIDE: dict[str, tuple[float, str]] = {
    "SK": (
        0.0,
        "Rates are set by Measure 2/2018 and have not moved since 2019. "
        "Applying the European average would add thirteen years of increase "
        "to a tariff that has demonstrably been flat for seven of them. "
        "COUNTER-ARGUMENT, deliberately recorded: this assumes the freeze "
        "holds another eight years, which would make fourteen unchanged in "
        "total — implausible on its face. A revision, when it comes, is "
        "likely to catch up at once. Treat zero as the low end of a real "
        "0-3%/yr range, not as certainty, and revisit if ŽSR reprices.",
    ),
}


def escalation_rate(country_code: str) -> float:
    """The escalation rate for one country — the European average unless a
    documented national deviation applies."""
    override = ESCALATION_OVERRIDE.get(country_code)
    return TAC_ESCALATION_PER_YEAR if override is None else override[0]


# ECB reference rates, snapshot date below. Nine calibrated countries publish
# in a non-euro currency, so this table is a calibration input in its own
# right rather than a formatting detail.
FX_SNAPSHOT = "2026-08-11"
FX_TO_EUR = {
    "EUR": 1.0,
    "CHF": 1.064,
    "CZK": 1 / 24.5,
    "DKK": 1 / 7.46,
    "GBP": 1.20,
    "HUF": 1 / 396.0,
    "NOK": 1 / 11.7,
    "PLN": 1 / 4.25,
    "RON": 1 / 5.08,
    "SEK": 1 / 11.2,
}


def to_eur(value: float, currency: str) -> float:
    """Native currency to EUR at the pinned FX snapshot."""
    return value * FX_TO_EUR[currency]


def escalate(
    value: float, basis_year: int, rate: float = TAC_ESCALATION_PER_YEAR
) -> float:
    """Carry a value from its document price basis to the evaluation year."""
    return value * (1.0 + rate) ** (TARGET_YEAR - basis_year)


def to_model_value(
    value: float, currency: str, basis_year: int, country_code: str
) -> float:
    """The number the database receives: EUR, at the evaluation year.

    Both conversions happen exactly once, here. calc_tac.py never sees a
    native currency or a price basis.
    """
    return escalate(to_eur(value, currency), basis_year, escalation_rate(country_code))

## Value record

One dataclass carrying a value and its provenance together. Values stay in
native currency at their document price basis; the EUR and evaluation-year
forms are derived properties, so the register remains checkable against the
source document.

In [ ]:
# --- value record ----------------------------------------------------------
@dataclass
class SV:
    """One calibrated value with its provenance.

    `value` is always as published, in `currency` at `basis_year`. The EUR
    and evaluation-year forms are derived, never stored — so the register
    stays checkable against the source document.
    """

    country_code: str
    parameter: str
    value: float | None
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: int | None = None
    note: str = ""
    low: float | None = None
    high: float | None = None

    def __post_init__(self):
        if self.status == ASSUMED and (self.low is None or self.high is None):
            raise ValueError(
                f"{self.country_code}.{self.parameter}: ASSUMED needs a band"
            )
        if self.status in (SOURCED, DERIVED) and not self.source_id:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: {self.status} needs a source"
            )
        if self.value is not None and self.basis_year is None:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: a value needs a price basis"
            )

    @property
    def eur(self) -> float | None:
        return None if self.value is None else to_eur(self.value, self.currency)

    @property
    def model_value(self) -> float | None:
        """EUR at the evaluation year — what the seed export writes."""
        if self.value is None:
            return None
        if self.unit == "factor":
            return self.value  # multipliers are dimensionless, never escalated
        return to_model_value(
            self.value, self.currency, self.basis_year, self.country_code
        )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]

# Every country carries a full parameter set so a NULL is explicit rather
# than absent: a missing row and a documented "not levied" must not look
# alike downstream.
PARAMETERS = [
    "b_day",
    "b_night",
    "gamma",
    "seat_km",
    "per_stop",
    "revenue_share",
    "fixed_per_train_km",
    "peak_multiplier",
    "congestion_surcharge_eur_km",
]

COUNTRIES = [
    "AT",
    "BE",
    "BG",
    "CH",
    "CY",
    "CZ",
    "DE",
    "DK",
    "EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IT",
    "LT",
    "LU",
    "LV",
    "MT",
    "NL",
    "NO",
    "PL",
    "PT",
    "RO",
    "SE",
    "SI",
    "SK",
    "UK",
]

# CY and MT have no railway at all — a positive fact, distinct from an
# uncalibrated country, and the reason the loader's default substitution
# never prices a real leg for them.
NO_RAILWAY_COUNTRIES = {"CY", "MT"}

values: list[SV] = []

## Calibrated values

One block per country. A country appears here only for terms it actually
levies — an absent term becomes an explicit NULL in the export, which the
loader reads as *not levied* rather than *not known*.

In [ ]:
# --- Calibrated per-country values --------------------------------------
#
# One block per country, in the order the summary table reads. Values are
# as published: native currency, document price basis. Conversion to EUR
# and to the evaluation year happens in the export cell, not here.

# AT — Austria (ÖBB-Infrastruktur)
values += [
    SV(
        "AT",
        "b_day",
        0.643,
        "EUR/train-km",
        SOURCED,
        source_id="AT-SNNB-2027",
        locator="Tab.11 §5.3.4",
        currency="EUR",
        basis_year=2027,
        note="Personenfernverkehr",
    ),
    SV(
        "AT",
        "congestion_surcharge_eur_km",
        1.6081,
        "EUR/train-km",
        SOURCED,
        source_id="AT-SNNB-2027",
        locator="§5.4 überlastete Schienenwege",
        currency="EUR",
        basis_year=2027,
        note="flat surcharge on declared overloaded sections; blanket application to peak-overlapping run shares is the model's conservative assumption (PEAK_BANDS)",
    ),
    SV(
        "AT",
        "gamma",
        0.002282,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="AT-SNNB-2027",
        locator="Tab.13 §5.3.4",
        currency="EUR",
        basis_year=2027,
    ),
]

# BE — Belgium (Infrabel)
values += [
    SV(
        "BE",
        "b_night",
        2.720182,
        "EUR/train-km",
        DERIVED,
        source_id="BE-NS-2027",
        locator="App F.2 sheets 2.1.1 / 2.1.2.3",
        currency="EUR",
        basis_year=2026,
        note="direct cost 2.142772 + off-peak high-density mark-up 0.577410; density class assumed High",
    ),
]

# BG — Bulgaria (NRIC)
values += [
    SV(
        "BG",
        "b_day",
        0.2495,
        "EUR/train-km",
        SOURCED,
        source_id="BG-NRIC-2026",
        locator="§I",
        currency="EUR",
        basis_year=2026,
    ),
    SV(
        "BG",
        "gamma",
        0.00083,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="BG-NRIC-2026",
        locator="§I",
        currency="EUR",
        basis_year=2026,
    ),
]

# CH — Switzerland (federal ordinances)
values += [
    SV(
        "CH",
        "b_day",
        2.5,
        "CHF/train-path-km",
        SOURCED,
        source_id="CH-NZV-BAV",
        locator="Art.1 Anhang 1",
        currency="CHF",
        basis_year=2026,
        note="line category A; quality factor 1.0 for treaty cross-border paths",
    ),
    SV(
        "CH",
        "gamma",
        0.0036,
        "CHF/gross-tonne-km",
        SOURCED,
        source_id="CH-NZV-BAV",
        locator="Art.1(3)(b)",
        currency="CHF",
        basis_year=2026,
        note="Basispreis Verschleiss proxy for the per-vehicle formula",
    ),
    SV(
        "CH",
        "peak_multiplier",
        2.0,
        "factor",
        SOURCED,
        source_id="CH-NZV-BAV",
        locator="Art.1 Anhang 1",
        currency="EUR",
        basis_year=2026,
        note="doubles the day-rate term on declared high-load sections during the commuter peak; blanket application to peak-overlapping run shares is the model's conservative assumption (PEAK_BANDS)",
    ),
    SV(
        "CH",
        "per_stop",
        2.0,
        "CHF/stop",
        SOURCED,
        source_id="CH-NZV",
        locator="Art.19a(4)",
        currency="CHF",
        basis_year=2026,
        note="capacity element of the path price, not a station charge — belongs here",
    ),
]

# CZ — Czechia (Správa železnic)
values += [
    SV(
        "CZ",
        "gamma",
        0.08163,
        "CZK/gross-tonne-km",
        SOURCED,
        source_id="CZ-NS-2027",
        locator="charging annex",
        currency="CZK",
        basis_year=2027,
        note="Px=1.0 passenger; kETCS=1.0, no discount claimed",
    ),
]

# DE — Germany (DB InfraGO)
values += [
    SV(
        "DE",
        "b_night",
        2.76,
        "EUR/train-km",
        ASSUMED,
        source_id="DE-INB-2026",
        locator="Anlage 5.3",
        currency="EUR",
        basis_year=2026,
        low=2.76,
        high=3.33,
        note="INB 2026 prints 3.33; the -17% BNetzA SPFV re-approval is applied but its decision document is not in the source set. Revert to 3.33 if it does not hold for Nacht.",
    ),
]

# DK — Denmark (Banedanmark)
values += [
    SV(
        "DK",
        "b_day",
        5.8,
        "DKK/train-km",
        SOURCED,
        source_id="DK-BEK-2024",
        locator="infrastrukturafgifter",
        currency="DKK",
        basis_year=2025,
        note="NS 2027 carries no numbers and refers to the executive order",
    ),
]

# EE — Estonia (Eesti Raudtee)
values += [
    SV(
        "EE",
        "b_day",
        0.74,
        "EUR/train-km",
        SOURCED,
        source_id="EE-TTJA-2026",
        locator="rates table",
        currency="EUR",
        basis_year=2026,
    ),
    SV(
        "EE",
        "gamma",
        0.00299,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="EE-TTJA-2026",
        locator="rates table",
        currency="EUR",
        basis_year=2026,
        note="international passenger mark-up is zero",
    ),
]

# ES — Spain (Adif)
values += [
    SV(
        "ES",
        "b_day",
        5.3181,
        "EUR/train-km",
        DERIVED,
        source_id="ES-BOE-2024",
        locator="Art.4",
        currency="EUR",
        basis_year=2023,
        note="Mode A 1.6767 + Mode B 3.6414, line type A (ES-BOE-2024 Art.4); standard-gauge night trains from FR stay on the HS network",
    ),
    SV(
        "ES",
        "seat_km",
        0.022014,
        "EUR/seat-km",
        SOURCED,
        source_id="ES-BOE-2024",
        locator="Art.5",
        currency="EUR",
        basis_year=2023,
        note="Madrid-Barcelona-Frontera surcharge, 2.2014 EUR per 100 seat-km",
    ),
]

# FI — Finland (Väylävirasto)
values += [
    SV(
        "FI",
        "gamma",
        0.002054,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="FI-NS-2027",
        locator="Tab.2 §5.3",
        currency="EUR",
        basis_year=2027,
        note="electric-supply-equipment charge excluded as energy",
    ),
]

# FR — France (SNCF Réseau)
values += [
    SV(
        "FR",
        "b_day",
        0.657,
        "EUR/train-km",
        SOURCED,
        source_id="FR-DRR-2027-A52",
        locator="App 5.2.2 UIC 2-6",
        currency="EUR",
        basis_year=2027,
    ),
    SV(
        "FR",
        "gamma",
        0.005705,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="FR-DRR-2027-A52",
        locator="App 5.2.2 UIC 2-6",
        currency="EUR",
        basis_year=2027,
        note="published as 5.705 EUR per 1000 CGT-km; RM shows '-' for night trains",
    ),
]

# GR — Greece (OSE)
values += [
    SV(
        "GR",
        "b_day",
        1.897,
        "EUR/train-km",
        DERIVED,
        source_id="GR-OSE-2026",
        locator="ch.6",
        currency="EUR",
        # NOT 2019: the 2.64 tariff is a 2019 base, but the 1.1975 factor is
        # the network statement's own inflation adjustment carrying it to the
        # NS 2026 year. The calibrated value is therefore already at 2026
        # money, and recording 2019 here would apply seven years of general
        # escalation on top of an inflation step the source has performed.
        basis_year=2026,
        note="2.64 (2019 base) x 1.1975 NS inflation to 2026 x 0.60 recovery",
    ),
    SV(
        "GR",
        "gamma",
        0.004024,
        "EUR/gross-tonne-km",
        DERIVED,
        source_id="GR-OSE-2026",
        locator="ch.6",
        currency="EUR",
        basis_year=2026,  # same NS inflation step as b_day above
        note="0.00560 (2019 base) x 1.1975 NS inflation to 2026 x 0.60 recovery",
    ),
]

# HR — Croatia (HŽ Infrastruktura)
values += [
    SV(
        "HR",
        "b_day",
        2.154,
        "EUR/train-km",
        DERIVED,
        source_id="HR-NS-2027",
        locator="§5.3",
        currency="EUR",
        basis_year=2027,
        note="T 2.10 (EuroNight) x L1 1.90 x Cvlkm 0.54; L1 assumed for international mainlines",
    ),
]

# HU — Hungary (MÁV)
values += [
    SV(
        "HU",
        "b_day",
        1143.0,
        "HUF/train-km",
        SOURCED,
        source_id="HU-NS-2627",
        locator="Annex 5.2-6",
        currency="HUF",
        basis_year=2027,
        note="path ensuring 9 + track category I passenger rate 1134",
    ),
    SV(
        "HU",
        "gamma",
        1.05,
        "HUF/gross-tonne-km",
        SOURCED,
        source_id="HU-NS-2627",
        locator="Annex 5.2-6",
        currency="HUF",
        basis_year=2027,
    ),
]

# IE — Ireland (Iarnród Éireann)
values += [
    SV(
        "IE",
        "b_day",
        1.9268,
        "EUR/train-km",
        SOURCED,
        source_id="IE-NS-2027",
        locator="ch.6.2/6.3",
        currency="EUR",
        basis_year=2027,
        note="fixed track access charge is franchise-only",
    ),
]

# IT — Italy (RFI)
values += [
    SV(
        "IT",
        "b_day",
        0.185,
        "EUR/train-km",
        SOURCED,
        source_id="IT-LISTINO",
        locator="Component A flat term",
        currency="EUR",
        basis_year=2027,
    ),
    SV(
        "IT",
        "b_night",
        2.31,
        "EUR/train-km",
        SOURCED,
        source_id="IT-LISTINO",
        locator="Component B Basic FOND Standard Notturno",
        currency="EUR",
        basis_year=2027,
        note="night band assumed 22:00-06:00, not confirmed in the Listino",
    ),
    SV(
        "IT",
        "gamma",
        0.002707,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="IT-LISTINO",
        locator="Component A speed class 150-175",
        currency="EUR",
        basis_year=2027,
        note="speed class taken from route average speed at runtime",
    ),
]

# LT — Lithuania (LTG Infra)
values += [
    SV(
        "LT",
        "gamma",
        0.0012,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="LT-LTG-2627",
        locator="tariff decision §5.3.2",
        currency="EUR",
        basis_year=2027,
    ),
]

# LU — Luxembourg (ACF/CFL)
values += [
    SV(
        "LU",
        "b_day",
        3.333,
        "EUR/train-km",
        DERIVED,
        source_id="LU-NS-2027",
        locator="§5.3.2",
        currency="EUR",
        basis_year=2026,
        note="cC 2.771 x alpha 1.1615 (>8 bodies) x beta 1.0355",
    ),
    SV(
        "LU",
        "fixed_per_train_km",
        0.05,
        "EUR/train-km",
        SOURCED,
        source_id="LU-NS-2027",
        locator="§5.3.2",
        currency="EUR",
        basis_year=2026,
        note="path administration on a regular timetable path",
    ),
]

# LV — Latvia (LDz)
values += [
    SV(
        "LV",
        "b_day",
        1.3,
        "EUR/train-km",
        SOURCED,
        source_id="LV-NS-2027",
        locator="§5.2",
        currency="EUR",
        basis_year=2026,
        note="international passenger within the EEA",
    ),
    SV(
        "LV",
        "gamma",
        0.00106848,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="LV-NS-2027",
        locator="§5.2",
        currency="EUR",
        basis_year=2026,
    ),
]

# NL — Netherlands (ProRail)
values += [
    SV(
        "NL",
        "b_day",
        2.2252,
        "EUR/train-km",
        SOURCED,
        source_id="NL-NS-2027",
        locator="train path service, class 601-3200 t",
        currency="EUR",
        basis_year=2027,
        note="weight class resolved from composition at runtime; all mark-ups are zero",
    ),
]

# NO — Norway (Bane NOR)
values += [
    SV(
        "NO",
        "b_day",
        9.94,
        "NOK/train-km",
        SOURCED,
        source_id="NO-NS-2027",
        locator="Tab.4 §5.3.3",
        currency="NOK",
        basis_year=2026,
        note="non-Oslo rate applied uniformly, conservative",
    ),
]

# PL — Poland (PKP PLK)
values += [
    SV(
        "PL",
        "b_day",
        8.01,
        "PLN/train-km",
        SOURCED,
        source_id="PL-PLK-A91",
        locator="Annex 9.1 SMK",
        currency="PLN",
        basis_year=2027,
        note="x WM(mass) x WK(line category), both 1.0 for NT-REF",
    ),
]

# PT — Portugal (Infraestruturas de Portugal)
values += [
    SV(
        "PT",
        "b_day",
        2.55,
        "EUR/train-km",
        SOURCED,
        source_id="PT-IP-2027",
        locator="§5.3 cat A Regular/Peak electric",
        currency="EUR",
        basis_year=2027,
    ),
    SV(
        "PT",
        "b_night",
        2.16,
        "EUR/train-km",
        SOURCED,
        source_id="PT-IP-2027",
        locator="§5.3 cat A Low electric",
        currency="EUR",
        basis_year=2027,
    ),
]

# RO — Romania (CFR SA)
values += [
    SV(
        "RO",
        "b_day",
        13.22,
        "RON/train-km",
        DERIVED,
        source_id="RO-CFR-A25",
        locator="Annex 25.a",
        currency="RON",
        basis_year=2024,
        note="Tc 9.562 + Ttsn 3.45 x [1 + (m-60) x 0.00014] at 600 t, class A; reproduces the worked example in Annex 26.a exactly",
    ),
]

# SE — Sweden (Trafikverket)
values += [
    SV(
        "SE",
        "b_day",
        4.98,
        "SEK/train-km",
        SOURCED,
        source_id="SE-NS-2027",
        locator="Annex 1B",
        currency="SEK",
        basis_year=2027,
    ),
    SV(
        "SE",
        "gamma",
        0.0218,
        "SEK/gross-tonne-km",
        SOURCED,
        source_id="SE-NS-2027",
        locator="Annex 1B",
        currency="SEK",
        basis_year=2027,
        note="passenger, mean axle load <=17 t",
    ),
]

# SI — Slovenia (SŽ-Infrastruktura)
values += [
    SV(
        "SI",
        "b_day",
        2.41,
        "EUR/train-km",
        DERIVED,
        source_id="SI-NS-2027",
        locator="§5.3",
        currency="EUR",
        basis_year=2027,
        note="C_P1 2.01 x PP 1.44 (R4) x PD 1.05 x PM 0.75 x PV 0.97 x Pl 1.09; ETCS incentive not claimed",
    ),
]

# SK — Slovakia (ŽSR)
values += [
    SV(
        "SK",
        "b_day",
        1.0661,
        "EUR/train-km",
        DERIVED,
        source_id="SK-ZSR-A52B",
        locator="Measure 2/2018 Annex 1",
        currency="EUR",
        basis_year=2019,
        note="U1 0.0691 + U2 0.997, track category 1",
    ),
    SV(
        "SK",
        "gamma",
        0.001102,
        "EUR/gross-tonne-km",
        SOURCED,
        source_id="SK-ZSR-A52B",
        locator="Measure 2/2018 Annex 1 U3",
        currency="EUR",
        basis_year=2019,
        note="published as 1.102 EUR per 1000 gtkm",
    ),
]

# UK — Great Britain (Network Rail)
values += [
    SV(
        "UK",
        "b_day",
        2.247,
        "GBP/train-km",
        DERIVED,
        source_id="UK-NR-CP7",
        locator="Default Passenger VUC",
        currency="GBP",
        basis_year=2024,
        note="(loco 127.05 + 10 x coach 23.45) pence per vehicle-mile / 1.609344",
    ),
]

## Night and peak bands

In [ ]:
# --- Night bands -----------------------------------------------------------
# Two mechanisms only: `none` and `time_band`. An earlier `segment` mode for
# Germany was dropped — the DE tariff is a band tariff like IT/BE/PT, and the
# SPFV rule is a band *widening*, not a separate mechanism: a train carrying
# night accommodation is priced Nacht over its entire German run. That
# widening is the boolean, evaluated from the composition rather than the
# timetable. CH sits in `none` deliberately: its 22:00-06:00 band belongs to
# electricity pricing, not track access.
NIGHT_COLUMNS = [
    "country_code",
    "night_mode",
    "night_band_start",
    "night_band_end",
    "night_full_if_accommodation",
]

NIGHT_BANDS = {
    "BE": ("time_band", "19:00", "05:59", False),
    "DE": ("time_band", "23:00", "06:00", True),
    "IT": ("time_band", "22:00", "06:00", False),
    "PT": ("time_band", "20:45", "06:00", False),
}

night_rows = []
for cc in COUNTRIES:
    mode, start, end, widen = NIGHT_BANDS.get(cc, ("none", "", "", False))
    night_rows.append(
        {
            "country_code": cc,
            "night_mode": mode,
            "night_band_start": start,
            "night_band_end": end,
            "night_full_if_accommodation": widen,
        }
    )

# A band tariff needs a band; a country without one must not carry stray hours.
for r in night_rows:
    if r["night_mode"] == "time_band":
        assert r["night_band_start"] and r["night_band_end"], r["country_code"]
    else:
        assert not r["night_band_start"] and not r["night_band_end"], r["country_code"]
    assert (r["country_code"] in NIGHT_BANDS) == (r["night_mode"] != "none")

# A night rate is only reachable through a band — otherwise it is dead data.
_banded = {cc for cc, v in NIGHT_BANDS.items()}
_with_night = {
    v.country_code for v in values if v.parameter == "b_night" and v.value is not None
}
assert _with_night <= _banded, (
    f"b_night without a band: {sorted(_with_night - _banded)}"
)


# --- Peak bands ------------------------------------------------------------
# AT's congestion surcharge and CH's peak factor formally apply only on
# declared overloaded sections. The model applies them on the peak-overlapping
# share of a run regardless, because a night train's morning approach into
# Wien Hbf or Zürich HB plausibly touches such a section, and pricing every
# unconfirmed approach at zero would understate cost in exactly the pattern
# night trains run. Weekday-only bands are priced at expected value
# (WEEKDAY_BLEND = 5/7) since the model has clock minutes, not service dates.
PEAK_COLUMNS = [
    "country_code",
    "band1_start",
    "band1_end",
    "band2_start",
    "band2_end",
    "weekdays_only",
]

PEAK_BANDS = {
    "AT": ("06:00", "09:00", "16:00", "19:00", True),
    "CH": ("06:00", "09:00", "16:00", "19:00", True),
}

peak_rows = [
    {
        "country_code": cc,
        "band1_start": b[0],
        "band1_end": b[1],
        "band2_start": b[2],
        "band2_end": b[3],
        "weekdays_only": b[4],
    }
    for cc, b in sorted(PEAK_BANDS.items())
]

# A peak term without a band would never be charged, and a band without a
# term would charge nothing — both are calibration errors, not tariff facts.
_peak_terms = {
    v.country_code
    for v in values
    if v.parameter in ("peak_multiplier", "congestion_surcharge_eur_km")
    and v.value is not None
}
assert _peak_terms == set(PEAK_BANDS), (
    f"peak terms {sorted(_peak_terms)} do not match bands {sorted(PEAK_BANDS)}"
)

## Passage charges

Crossings charged per traverse rather than per km.

In [ ]:
# --- Passage charges -------------------------------------------------------
# Fixed charges tied to a specific crossing, levied per traverse rather than
# per km. Kept as their own entity keyed by crossing (extensible to
# Fehmarnbelt in the 2032 scenario) rather than folded into country
# attributes, because the charging entity is the crossing operator, not the
# country. Øresund is one polygon with two charge rows: each infrastructure
# manager bills its half.
PASSAGE_COLUMNS = [
    "crossing_id",
    "charged_by",
    "parameter",
    "value",
    "unit",
    "currency",
    "basis_year",
    "source_id",
    "locator",
    "note",
]

passage_rows = [
    {
        "crossing_id": "STOREBAELT",
        "charged_by": "Banedanmark",
        "parameter": "fixed_per_train",
        "value": 4876.73,
        "unit": "DKK/train",
        "currency": "DKK",
        "basis_year": 2025,
        "source_id": "DK-BEK-2024",
        "locator": "infrastrukturafgifter",
        "note": "excl. VAT",
    },
    {
        "crossing_id": "OERESUND_DK",
        "charged_by": "Banedanmark",
        "parameter": "fixed_per_train",
        "value": 2592.02,
        "unit": "DKK/train",
        "currency": "DKK",
        "basis_year": 2025,
        "source_id": "DK-BEK-2024",
        "locator": "infrastrukturafgifter",
        "note": "Danish part",
    },
    {
        "crossing_id": "OERESUND_SE",
        "charged_by": "Trafikverket",
        "parameter": "fixed_per_train",
        "value": 0.0,
        "unit": "SEK/train",
        "currency": "SEK",
        "basis_year": 2027,
        "source_id": "SE-NS-2027",
        "locator": "Annex 1B",
        "note": "zero is a tariff fact, not missing data: the passage charge "
        "applies to freight only. Regular SE track and path charges "
        "still apply on the link",
    },
    {
        "crossing_id": "CHANNEL_TUNNEL",
        "charged_by": "Getlink",
        "parameter": "fixed_per_train",
        "value": 4039.0,
        "unit": "EUR/train one-way",
        "currency": "EUR",
        "basis_year": 2020,
        "source_id": "CT-GETLINK-2026",
        "locator": "Annexe 4 Offer 1",
        "note": "night trains run at 120 km/h off-peak by definition, so the "
        "off-peak row is the night-train tariff; billed half EUR half "
        "GBP, combined at the statement's own rate",
    },
    {
        "crossing_id": "CHANNEL_TUNNEL",
        "charged_by": "Getlink",
        "parameter": "per_passenger",
        "value": 18.35,
        "unit": "EUR/passenger one-way",
        "currency": "EUR",
        "basis_year": 2020,
        "source_id": "CT-GETLINK-2026",
        "locator": "Annexe 4 Offer 1",
        "note": "couples the crossing cost to the demand model rather than to "
        "routing — evaluated against the segment's passenger load in "
        "the traffic pre-pass",
    },
]

# Crossing polygons. Detection happens at routing time: the first trip leg
# intersecting a polygon owns the crossing. Coarse rectangles are sufficient —
# they only need to catch the fixed link and miss everything else.
PASSAGE_GEOMETRIES = {
    "STOREBAELT": [
        [10.75, 55.25],
        [11.05, 55.25],
        [11.05, 55.45],
        [10.75, 55.45],
        [10.75, 55.25],
    ],
    "OERESUND": [
        [12.60, 55.53],
        [12.95, 55.53],
        [12.95, 55.66],
        [12.60, 55.66],
        [12.60, 55.53],
    ],
    "CHANNEL_TUNNEL": [
        [1.45, 50.85],
        [1.95, 50.85],
        [1.95, 51.15],
        [1.45, 51.15],
        [1.45, 50.85],
    ],
}

# Øresund is deliberately one polygon behind two charge rows — each IM bills
# its half — so the mapping is declared rather than inferred from the id.
CHARGE_TO_POLYGON = {
    "STOREBAELT": "STOREBAELT",
    "OERESUND_DK": "OERESUND",
    "OERESUND_SE": "OERESUND",
    "CHANNEL_TUNNEL": "CHANNEL_TUNNEL",
}

_charge_ids = {r["crossing_id"] for r in passage_rows}
assert _charge_ids == set(CHARGE_TO_POLYGON), (
    f"charge rows and polygon mapping disagree: {_charge_ids ^ set(CHARGE_TO_POLYGON)}"
)
# A charge with no polygon can never be detected; a polygon with no charge
# would be detected and then cost nothing. Both are calibration errors.
assert set(CHARGE_TO_POLYGON.values()) == set(PASSAGE_GEOMETRIES), (
    f"polygons and charges disagree: "
    f"{set(CHARGE_TO_POLYGON.values()) ^ set(PASSAGE_GEOMETRIES)}"
)

## Export

In [ ]:
# --- Export ----------------------------------------------------------------


def write_csv(path: Path, columns: list[str], rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {path.name}: {len(rows)} rows")


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """A calibration table under `data/` — see the seed export cell for the
    database-facing counterpart."""
    write_csv(DATA_DIR / name, columns, rows)


# Fill the grid: every country carries every parameter, so a NULL is a
# statement ("not levied here") rather than an absence.
calibrated = {(v.country_code, v.parameter): v for v in values}
assert len(calibrated) == len(values), "duplicate country/parameter pair"

full_grid: list[SV] = []
for cc in COUNTRIES:
    for p in PARAMETERS:
        hit = calibrated.get((cc, p))
        if hit is not None:
            full_grid.append(hit)
        elif cc in NO_RAILWAY_COUNTRIES:
            full_grid.append(SV(cc, p, None, "", NO_RAILWAY, note="no railway network"))
        else:
            full_grid.append(SV(cc, p, None, "", MISSING))

write_data("tac_components.csv", SV_FIELDS, [asdict(v) for v in full_grid])
write_data("tac_night_mode.csv", NIGHT_COLUMNS, night_rows)
write_data("tac_peak_bands.csv", PEAK_COLUMNS, peak_rows)
write_data("passage_charges.csv", PASSAGE_COLUMNS, passage_rows)

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"crossing_id": cid},
            "geometry": {"type": "Polygon", "coordinates": [coords]},
        }
        for cid, coords in sorted(PASSAGE_GEOMETRIES.items())
    ],
}
with open(DATA_DIR / "passage_geometries.geojson", "w", encoding="utf-8") as fh:
    json.dump(geojson, fh, indent=1)
    fh.write("\n")
print(f"  passage_geometries.geojson: {len(geojson['features'])} polygons")

by_status: dict[str, int] = {}
for v in full_grid:
    by_status[v.status] = by_status.get(v.status, 0) + 1
print()
for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
    print(f"    {s:11} {n:4}")

# A derivation that already carries the source's own inflation step must
# record the year it was carried TO, not the year of the underlying tariff —
# otherwise the escalation applies those years a second time. Greece is the
# live case (a 2019 tariff the network statement inflates to 2026, which was
# recorded as basis 2019 and would have been escalated 23% too far).
#
# The invariant is deliberately simple: if a note claims an inflation or
# indexation step, it must name the year that step lands on, and that year
# must be the price basis. Being forced to write "to 2026" is what makes the
# author confront the direction.
_INFLATION_WORDS = ("inflation", "indexed", "indexation", "escalat")
_suspect = [
    f"{v.country_code}.{v.parameter}"
    for v in values
    if v.value is not None
    and any(w in v.note.lower() for w in _INFLATION_WORDS)
    and f"to {v.basis_year}" not in v.note
]
assert not _suspect, (
    "note claims an inflation step but does not say it lands on the recorded "
    f"price basis — double-counting risk: {_suspect}"
)

# Every source a value leans on must exist in the register written by 01 —
# a dangling source_id is provenance that looks real and is not.
_register_path = DATA_DIR / "sources_register.csv"
if _register_path.exists():
    with open(_register_path, encoding="utf-8") as fh:
        known = {r["source_id"] for r in csv.DictReader(fh)}
    cited = {v.source_id for v in values if v.source_id}
    cited |= {r["source_id"] for r in passage_rows if r["source_id"]}
    assert cited <= known, f"cited but not in the register: {sorted(cited - known)}"
    print(f"\n  provenance: {len(cited)} sources cited, all present in the register")
else:
    print(
        "\n  NOTE: run 01_source_extraction.ipynb first to enable the provenance check"
    )

## Seed export

What `db/dev/seed.py` reads. The same values as `data/` above, after both
conversions and pivoted one row per country — the database receives plain
EUR at the evaluation year and never a currency or a price basis.


In [ ]:
# --- Seed export -----------------------------------------------------------
# What the database receives. Distinct from `data/` above, which is the
# calibration record: native currency, document price basis, one row per
# country and parameter, checkable line by line against the source.
#
# The seed CSVs are the same values after both conversions (FX, then price
# basis to the evaluation year) and pivoted to one row per country, so
# db/dev/seed.py inserts a number and never a unit. An empty cell is a NULL,
# and a NULL means "this country does not levy this term" — the loader's
# substitution of the EU-median default group is a separate, deliberate
# decision made at load time, never here.

SEED_DIR = DATA_DIR.parent / "seed"
SEED_DIR.mkdir(exist_ok=True)

# Human-readable crossing names — a database display concern rather than a
# calibrated value, which is why they live here and not with the charges.
PASSAGE_NAMES = {
    "STOREBAELT": "Storebælt fixed link",
    "OERESUND_DK": "Øresund fixed link (Danish part)",
    "OERESUND_SE": "Øresund fixed link (Swedish part)",
    "CHANNEL_TUNNEL": "Channel Tunnel",
}

TRACK_TAC_COLUMNS = [
    "country_code",
    "track_tac_b_day",
    "track_tac_b_night",
    "track_tac_gamma",
    "track_tac_seat_km",
    "track_tac_per_stop",
    "track_tac_revenue_share",
    "track_tac_fixed_per_train_km",
    "track_tac_peak_multiplier",
    "track_tac_congestion_surcharge_eur_km",
    "track_tac_night_mode",
    "track_tac_night_band_start",
    "track_tac_night_band_end",
    "track_tac_night_full_if_accommodation",
    "track_tac_peak_band1_start",
    "track_tac_peak_band1_end",
    "track_tac_peak_band2_start",
    "track_tac_peak_band2_end",
    "track_tac_peak_weekdays_only",
    "source_id",
    "change_log",
]

# Eight decimals, matching the widest TAC column in db/schema.py: gamma and
# seat_km are of order 1e-3, so a coarser rounding would quantise a real
# per-country difference into noise.
_SEED_NDIGITS = 8


def _fmt(value: float) -> str:
    """Fixed-point, trailing zeros stripped — no scientific notation, which
    the CSV reader in seed.py would still parse but nobody can eyeball."""
    return f"{value:.{_SEED_NDIGITS}f}".rstrip("0").rstrip(".")


def _seed_value(sv: "SV | None") -> str:
    """One component as the database receives it: EUR at the evaluation
    year, or empty for a NULL."""
    if sv is None or sv.model_value is None:
        return ""
    return _fmt(sv.model_value)


night_by_cc = {r["country_code"]: r for r in night_rows}
peak_by_cc = {r["country_code"]: r for r in peak_rows}

track_tac_seed = []
for cc in COUNTRIES:
    night = night_by_cc[cc]
    peak = peak_by_cc.get(cc, {})
    cited = sorted(
        {v.source_id for v in values if v.country_code == cc and v.source_id}
    )
    # One group-level source FK per country (input_params.track_infrastructures
    # .track_tac_src): a country's terms come from its network statement, and
    # 27 of 28 calibrated countries cite exactly one document. Where more are
    # involved the extras are named in change_log rather than lost — the
    # per-value record stays tac_components.csv and TAC_CALIBRATION.md.
    extra = (
        f"TAC components also sourced from {', '.join(cited[1:])}."
        if len(cited) > 1
        else ""
    )
    track_tac_seed.append(
        {
            "country_code": cc,
            **{
                f"track_tac_{p}": _seed_value(calibrated.get((cc, p)))
                for p in PARAMETERS
            },
            "track_tac_night_mode": night["night_mode"],
            "track_tac_night_band_start": night["night_band_start"],
            "track_tac_night_band_end": night["night_band_end"],
            "track_tac_night_full_if_accommodation": night[
                "night_full_if_accommodation"
            ],
            "track_tac_peak_band1_start": peak.get("band1_start", ""),
            "track_tac_peak_band1_end": peak.get("band1_end", ""),
            "track_tac_peak_band2_start": peak.get("band2_start", ""),
            "track_tac_peak_band2_end": peak.get("band2_end", ""),
            "track_tac_peak_weekdays_only": peak.get("weekdays_only", False),
            "source_id": cited[0] if cited else "",
            "change_log": extra,
        }
    )

# The fallback group. A country the model routes through but the register
# has no b_day/b_night/gamma for is priced from here rather than for free —
# see the loader's group-null rule (adapters/data_loader_from_db.py): the
# substitution happens only when ALL THREE rate terms are absent, so a
# country that levies gamma alone keeps its documented "no train-km term"
# rather than acquiring a median one.
#
# Median, not mean: the calibrated spread runs from 0.21 (BG) to 6.94
# (ES) EUR/train-km, and a mean over that is pulled up by a handful of
# high-cost networks into a number no European country actually charges.
# Only the three rate terms are defaulted — seat_km, per_stop,
# fixed_per_train_km, revenue_share and the peak terms are documented
# national particularities, and giving every uncalibrated country a
# Spanish seat surcharge or a Swiss stop charge would invent tariff
# structure rather than fill a gap.
DEFAULT_TAC_PARAMETERS = ["b_day", "b_night", "gamma"]


def _median(xs: list[float]) -> float:
    ordered = sorted(xs)
    mid = len(ordered) // 2
    if len(ordered) % 2:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2.0


default_tac = {"country_code": "_default", "track_tac_night_mode": "none"}
for p in DEFAULT_TAC_PARAMETERS:
    levied = [
        v.model_value for v in values if v.parameter == p and v.model_value is not None
    ]
    default_tac[f"track_tac_{p}"] = _fmt(_median(levied))
    print(f"  default {p}: median of {len(levied)} calibrated countries")

write_csv(SEED_DIR / "track_tac_default.csv", TRACK_TAC_COLUMNS, [default_tac])

write_csv(SEED_DIR / "track_tac.csv", TRACK_TAC_COLUMNS, track_tac_seed)

# Passage charges: one row per charge entity, the two per-traverse components
# side by side rather than stacked, each already in EUR at the evaluation
# year. Geometry travels with the row (OERESUND_DK and _SE legitimately carry
# the same polygon — each IM bills its half of one crossing).
PASSAGE_SEED_COLUMNS = [
    "passage_id",
    "passage_name",
    "passage_fixed_eur",
    "passage_per_passenger_eur",
    "source_id",
    "passage_geom",
]

passage_seed: dict[str, dict] = {}
for row in passage_rows:
    pid = row["crossing_id"]
    entry = passage_seed.setdefault(
        pid,
        {
            "passage_id": pid,
            "passage_name": PASSAGE_NAMES[pid],
            "passage_fixed_eur": 0.0,
            "passage_per_passenger_eur": 0.0,
            "source_id": row["source_id"],
            "passage_geom": json.dumps(
                {
                    "type": "Polygon",
                    "coordinates": [PASSAGE_GEOMETRIES[CHARGE_TO_POLYGON[pid]]],
                }
            ),
        },
    )
    column = (
        "passage_fixed_eur"
        if row["parameter"] == "fixed_per_train"
        else "passage_per_passenger_eur"
    )
    # No country code: a crossing operator is not a national tariff regime,
    # so no per-country escalation override can apply to it.
    entry[column] = round(
        to_model_value(row["value"], row["currency"], row["basis_year"], ""),
        _SEED_NDIGITS,
    )

write_csv(
    SEED_DIR / "passage_charges.csv",
    PASSAGE_SEED_COLUMNS,
    [passage_seed[pid] for pid in sorted(passage_seed)],
)

# The source register, reduced to what input_params.sources stores. Only the
# documents actually cited are seeded: an unused register row is a research
# note, not provenance any value points at.
SOURCE_SEED_COLUMNS = ["source_id", "source_description", "source_url", "source_date"]

cited_ids = {v.source_id for v in values if v.source_id}
cited_ids |= {r["source_id"] for r in passage_rows if r["source_id"]}

with open(DATA_DIR / "sources_register.csv", encoding="utf-8") as fh:
    register = {r["source_id"]: r for r in csv.DictReader(fh)}

source_seed = [
    {
        "source_id": sid,
        "source_description": (
            f"{register[sid]['title']} — {register[sid]['publisher']} "
            f"({register[sid]['pub_year']})"
        ),
        "source_url": register[sid]["url_or_file"],
        "source_date": register[sid]["date_accessed"],
    }
    for sid in sorted(cited_ids)
]
write_csv(SEED_DIR / "sources.csv", SOURCE_SEED_COLUMNS, source_seed)

# Every seeded country must be one the database knows, and every night or
# peak band must survive the pivot — a silently dropped band would price a
# whole country at its day rate without failing anything.
assert {r["country_code"] for r in track_tac_seed} == set(COUNTRIES)
assert sum(
    1 for r in track_tac_seed if r["track_tac_night_mode"] == "time_band"
) == len(NIGHT_BANDS)
assert sum(1 for r in track_tac_seed if r["track_tac_peak_band1_start"]) == len(
    PEAK_BANDS
)

## Document generation

`TAC_CALIBRATION.md` is an output like the CSVs. Judgement prose lives in the
template cell; every computed value is injected from live state, so document
and data cannot drift apart.

In [ ]:
# --- TAC_CALIBRATION.md generation -----------------------------------------
# The document is a notebook output, like the CSVs. Prose that carries
# judgement lives in the template below verbatim; every computed value —
# summary table, EUR conversions, escalation factors, source list — is
# injected from live notebook state, so the document and the data it
# describes cannot drift apart.

REFERENCE_TRAIN_T = 600.0  # 1 loco + 10 coaches, ~300 m, 500 places

_CUR_DECIMALS = {"HUF": 0}


def _fmt_native(v: float, currency: str) -> str:
    if currency in _CUR_DECIMALS:
        return f"{v:,.{_CUR_DECIMALS[currency]}f}"
    if abs(v) < 0.01:
        return f"{v:.6f}".rstrip("0")
    if abs(v) < 1:
        return f"{v:.4f}".rstrip("0")
    return f"{v:,.4f}".rstrip("0").rstrip(".")


def _fmt_eur(v: float) -> str:
    return (
        f"{v:.6f}".rstrip("0") if abs(v) < 0.01 else f"{v:.4f}".rstrip("0").rstrip(".")
    )


def _get(cc: str, param: str) -> SV | None:
    v = calibrated.get((cc, param))
    return v if v is not None and v.value is not None else None


def _reference_cost(cc: str) -> float | None:
    """Indicative EUR/train-km for the reference train, at the evaluation
    year — the rate term plus the gross-tonne-km term at 600 t."""
    rate = _get(cc, "b_night") or _get(cc, "b_day")
    gamma = _get(cc, "gamma")
    if rate is None and gamma is None:
        return None
    total = 0.0
    if rate is not None:
        total += rate.model_value
    if gamma is not None:
        total += gamma.model_value * REFERENCE_TRAIN_T
    return total


# --- summary table ---
_summary_lines = []
for cc in COUNTRIES:
    rate = _get(cc, "b_night") or _get(cc, "b_day")
    gamma = _get(cc, "gamma")
    if rate is None and gamma is None:
        continue
    anchor = rate or gamma
    label = "b_night" if (rate is not None and rate.parameter == "b_night") else "b_day"
    nat = f"{_fmt_native(rate.value, rate.currency)} {rate.currency}" if rate else "—"
    eur = _fmt_eur(rate.eur) if rate else "—"
    gnat = (
        f"{_fmt_native(gamma.value, gamma.currency)} {gamma.currency}" if gamma else "—"
    )
    geur = _fmt_eur(gamma.eur) if gamma else "—"
    _summary_lines.append(
        f"| {cc} | `{label}` | {nat} | {eur} | {gnat} | {geur} | "
        f"{anchor.basis_year} | **{_reference_cost(cc):.2f}** |"
    )
SUMMARY_TABLE_ROWS = "\n".join(_summary_lines)

# --- escalation factor table ---
# Countries carrying a national deviation are counted separately — folding
# them into the headline table would imply they move with the average.
_basis_counts: dict[int, int] = {}
for v in values:
    if (
        v.value is not None
        and v.unit != "factor"
        and v.country_code not in ESCALATION_OVERRIDE
    ):
        _basis_counts[v.basis_year] = _basis_counts.get(v.basis_year, 0) + 1

_esc_lines = []
for by in sorted(_basis_counts):
    yrs = TARGET_YEAR - by
    cells = " | ".join(
        f"{(1 + r) ** yrs:.3f}"
        if r != TAC_ESCALATION_PER_YEAR
        else f"**{(1 + r) ** yrs:.3f}**"
        for r in (
            TAC_ESCALATION_LOW,
            0.025,
            TAC_ESCALATION_PER_YEAR,
            TAC_ESCALATION_HIGH,
        )
    )
    _esc_lines.append(f"| {by} | {_basis_counts[by]} | {yrs} | {cells} |")
ESCALATION_TABLE_ROWS = "\n".join(_esc_lines)

_n_esc = sum(_basis_counts.values())
_escalatable = [v for v in values if v.value is not None and v.unit != "factor"]
_mean_uplift = sum(
    (1 + escalation_rate(v.country_code)) ** (TARGET_YEAR - v.basis_year)
    for v in _escalatable
) / len(_escalatable)

# Countries priced off the European average, and the documented exceptions.
OVERRIDE_ROWS = "\n".join(
    f"| {cc} | {rate:.0%}/yr | {reason} |"
    for cc, (rate, reason) in sorted(ESCALATION_OVERRIDE.items())
)
_n_override_values = sum(
    1 for v in _escalatable if v.country_code in ESCALATION_OVERRIDE
)

# --- FX table ---
FX_TABLE_ROWS = "\n".join(
    f"| {cur} | {1 / rate:,.3f} {cur} = 1 EUR | {rate:.6f} |"
    for cur, rate in sorted(FX_TO_EUR.items())
    if cur != "EUR"
)
_fx_used = sorted({v.currency for v in values if v.currency != "EUR"})

In [ ]:
COUNTRY_PROSE = r"""
### AT — Austria (ÖBB-Infrastruktur)

**Formula:** `TAC = trkm × 0.643 + gtkm × 0.002282` (long-distance passenger, excl. 20 % VAT)

**Values (TT2027):** train-km component z = 0.643 EUR/trkm (Personenfernverkehr); gross-tonne-km component btk = 0.002282 EUR/gtkm. Congestion surcharge 1.6081 EUR/trkm applies only on declared overloaded sections → assumed not applicable. No market markups in TT2027 (planned reintroduction 2029: preview +1.12 EUR/trkm commercial passenger — flag for the 2032 scenario sensitivity).

**Assumptions:** none needed beyond congestion exclusion; gross weight = full consist incl. locos.

**Source:** SNNB 2027, ch. 5.3.4 (Tabellen 11–13); publication 2025, price basis TT2027.
**On disk:** `AT-SNNB-2027.pdf`
**IRG cross-check:** survey 2025: 0.649 / 0.002129 — consistent (small yearly drift). ✓

### BE — Belgium (Infrabel)

**Formula:** `TAC = trkm × (DC_line + RB_markup(density, period))`

**Values (1 Jan 2026, 6-decimal indexation):** DC_line = 2.142772 EUR/trkm (all trains). Ramsey–Boiteux markup applies to loaded runs of open-access passenger (HkvNPso); night trains run in *off-peak* (weekdays 19:00–05:59) and *weekend night* periods. Off-peak markup by line density class: 0.155858 (very low) … 1.028466 (very high/NSL) EUR/trkm.

**Mechanism & assumptions:** period class resolved **per leg from the schedule** (off-peak weekdays 19:00–05:59, weekend night 19:00–05:59, weekend day 06:00–18:59, normal 09:00–14:59, peak 06:00–08:59/15:00–18:59, NSL hyper-peak on the North–South Link) — evening departures before 19:00 and arrivals after 06:00 price at the day/peak coefficients on those legs (peak coefficients 1.63–10.75, NSL 16.44 EUR/trkm). Density class **High** assumed → off-peak markup 0.577410 EUR/trkm (mainline arteries are high/very-high density; conservative without per-section density data); indicative off-peak total 2.720 EUR/trkm. Direct cost catenary 17.22 EUR/MWh excluded (energy).

**Source:** NS 2027 (version 30 Jun 2026) §5.3 + Appendix F.2 workbook (sheets 2.1.1, 2.1.2.3).
**On disk:** `BE-NS-2027.pdf`
**IRG cross-check:** survey 2025 DC 2.076386 — matches indexation chain 2023→2026. ✓

### BG — Bulgaria (NRIC)

**Formula:** `TAC = trkm × 0.2495 + gtkm × 0.00083` (passenger)

**Values (effective 1 Feb 2026):** 0.2495 EUR/trkm + 0.00083 EUR/gtkm. Charge for use of power-supply equipment (17.29 EUR) excluded (energy; unit ambiguous in doc).

**Source:** NRIC "Charges and Prices" (Annex 5.3.2, v.06, 18 Mar 2026), Section I.
**On disk:** `BG-NRIC-2026.pdf`
**IRG cross-check:** survey has BG per-gtkm structure — consistent. ✓

### CH — Switzerland (SBB Infrastruktur et al.; federal ordinances)

**Formula:** `TAC = tpkm × base(line cat) × quality_factor × peak(t) + gtkm × wear + 2 CHF × stops + revenue_share × CH_revenues [CHF]`

**Values (NZV status 1 Jan 2026; NZV-BAV status 1 Feb 2026):**
- Base path price: cat A 2.50 / cat B 1.15 / cat C 1.15 / cat D 0.70 CHF/tpkm (Anhang 1 assigns lines).
- Quality factor: 1.0 for cross-border passenger under state treaty (cat B); 0.4 for non-concession passenger (cat C); 1.25 concession long-distance.
- Peak factor `peak(t)` = 2 on listed high-load standard-gauge sections Mon–Fri 06:00–09:00 and 16:00–19:00, else 1 — **applied per leg from the schedule** (a night train arriving Zurich/Basel after 06:00 pays it on the approach legs; departures before 19:00 likewise).
- Long-train rebate: −0.01 CHF/tpkm per metre of trailing load above 500 m (runtime from composition length).
- Wear price (Basispreis Verschleiss): vehicle-specific formula on standard gauge (Anhang 1a–1c); simplified proxy 0.0036 CHF/gtkm (the directly applicable rate per Art. 1(3)(b)).
- **Haltezuschlag: 2 CHF per ordered stop, incl. origin/destination stops** (NZV Art. 19a(4), NZV-BAV Art. 2) — this is a *capacity price element of the train-path price* (each stop consumes path capacity), not a station-usage fee; platform provision itself is part of the CH basic services with no separate station charge. → **included here**, no double-counting risk with the later station model.
- Contribution margin (Deckungsbeitrag), **primary mode = revenue-based** since traffic revenues are a model input: `revenue_share × CH-attributable traffic revenues` (concession / eidg. Bewilligung services, Art. 20(1bis, 2, 6) NZV; share set by the authority, observed range 0–21 % per IRG survey — concrete value for international night trains **MISSING → scenario parameter**, already first-class in the calib model). Fallback mode (non-concession): 0.0027 CHF per offered place-km, loaded runs only.
- Electricity 12–14 Rp./kWh excluded (energy).

**Assumptions:** international night trains priced as cross-border-treaty paths → quality factor 1.0 (conservative vs. 0.4); line cat A on main corridors (conservative; much of the network is B → base+wear ≈ 3.3–4.7 CHF/tpkm for the reference train before margin and stops); wear via the 0.0036 CHF/gtkm proxy instead of the per-vehicle formula; Haltezuschlag applied at all commercial stops (conservative — formally only stops on the published mixed-traffic list are charged).

**Source:** SR 742.122 (NZV) Art. 19a, 20, 20a; SR 742.122.4 (NZV-BAV) Art. 1, 2, Anhänge.
**On disk:** `CH-NZV.pdf`, `CH-NZV-BAV.pdf`
**IRG cross-check:** survey 2025 mirrors all elements (2.50/1.15/0.70, factors, 0.0027, 0.0036). ✓

### CZ — Czechia (Správa železnic)

**Formula:** `TAC = L × ZI × M × Px × kETCS` (Cs component; CPK stop component deferred → §5)

**Values:** ZI = 0.08163 CZK/gtkm (1 Jan–11 Dec 2027; 0.07956 for 13–31 Dec 2026). Px = 1.00 (passenger, P1). ZRP (traffic management) = 0.00000 CZK/km.

**Assumptions:** kETCS = 1.0 (no ETCS discount claimed — conservative; discount value not cleanly extractable from the NS text). Capacity-allocation price (k1 + k2·L + k3·days) excluded as minor admin.

**Source:** NS 2027 (SŽ, EN web version), charging annex part II/III.
**On disk:** `CZ-NS-2027.pdf`
**IRG cross-check:** survey 2025 lists composite 30.34 CZK/trkm passenger ≈ 0.0796 × ~380 t — consistent with ZI × M structure. ✓

### DE — Germany (DB InfraGO)

**Formula:** `TAC = trkm × 2.76` (SPFV market segment *Nacht*, flat; −17 % applied)

**Values:** documented rate (INB 2026 Anlage 5.3, valid from 14 Dec 2025): Nacht = 3.33 EUR/trkm. **Calibrated value = 3.33 × 0.83 = 2.76 EUR/trkm**, applying the BNetzA SPFV −17 % re-approval (22 Jul 2026, prior-session finding) — **marked as assumption** since the re-approval decision itself is not among the uploaded documents (BK10-25-0067 Anlage 2 = approved consolidated INB 2026 text still showing 3.33). Revert to 3.33 if the re-approval does not hold for the Nacht segment.

Segment assignment (drives pro-rata mechanism): trains running 23:00–06:00, **or** trains fully traversing the night window (incl. foreign route portions) carrying at least one couchette/sleeper — then the *entire German run* (also before 23:00 / after 06:00) is priced as Nacht. Partial-window trains without sleepers: pro-rata by travel-time share within 23:00–06:00, applied per leg from the schedule.

**Source:** INB 2026 Anlage 5.3 (Redaktionsstand 12 Dec 2025), segment definition Ziffer 5.3.2.5; BNetzA BK10-25-0067 Anlage 2.
**On disk:** `DE-INB-2026.pdf`, `DE-BNETZA-BK10.pdf`
**IRG cross-check:** DE survey sheet consistent with segment logic. ✓

### DK — Denmark (Banedanmark)

**Formula:** `TAC = trkm × 5.80 DKK + Storebælt: 4,876.73 DKK/train + Øresund (DK part): 2,592.02 DKK/train` (passenger, excl. VAT)

**Values:** 2025 executive order (bekendtgørelse 2024/1351); NS 2027 contains no numbers and points to the current order (annually indexed).

**Assumptions:** 2025 rates carried as best available; `fixed_per_crossing` triggered by route touching the respective link.

**Source:** NS 2027 §5.2/5.3; rates per Executive Order on infrastructure charges (via IRG survey capture of the order).
**On disk:** `DK-NS-2027.pdf` — rates via `IRG-TAC-2025.xlsx` (the 2024 order itself is **not on disk**)
**IRG cross-check:** survey 2025: 0.78 EUR/trkm ≈ 5.80 DKK. ✓

### EE — Estonia (AS Eesti Raudtee)

**Formula:** `TAC = trkm × 0.74 + gtkm × 0.00299` (base MAP)

**Values (TT 2025/26):** base 0.74 EUR/trkm + 0.00299 EUR/gtkm. Markup for *domestic* passenger service: +1.45 EUR/trkm +0.00555 EUR/gtkm; markup for **international** passenger service: 0.

**Assumptions:** night train = international passenger → base only. Edelaraudtee network (if touched) not calibrated (MISSING; minor).

**Source:** TTJA published rates 2025/2026; EVR NS 2026.
**On disk:** `EE-TTJA-2026.pdf`
**IRG cross-check:** survey EE sheet empty ("no data") — TTJA page is the better source. ✓ (one-sided)

### ES — Spain (Adif / Adif AV)

**Formula:** `TAC = trkm × (canon Mode A + canon Mode B) [+ seat_km × surcharge on type A lines]`, service type VL1, line type selected per leg from infrastructure max speed

**Values (canon regulation, rates in force since 2023, consolidated 2024):**
- Lines *other than type A* (conventional): Mode A = 0.7273 + Mode B = 1.0030 → **1.7303 EUR/trkm**.
- **Type A (high-speed) lines:** Mode A = 1.6767 + Mode B = 3.6414 → **5.3181 EUR/trkm**, plus seat surcharge per 100 seat-km: Madrid–Barcelona–Frontera 2.2014; Madrid–Toledo–Sevilla–Málaga 1.0809; other type-A lines 0.5404 EUR. For a 500-place train on the French-border corridor: +11.0 EUR/trkm → ≈ 16.3 EUR/trkm total.
- Mode C (traction-electricity transformation/distribution) excluded (energy).

**Mechanism & assumptions:** line type A vs non-A resolved per leg from infra max speed (≥ 250 km/h → type A). **Standard-gauge night trains entering from France realistically stay on the HS network** (conventional network is Iberian gauge; gauge-changing stock would be required otherwise) → default cross-border ES routing prices as type A, corridor surcharge 2.2014 (Barcelona entry) unless routed otherwise. VL1 = commercial long-distance. Under/over-use cancellation surcharges not modelled.

**Source:** BOE-A-2024-22140 (Reglamento de cánones, consolidated), Arts. 3–5; Adif NS 2027 ch. 5.3.
**On disk:** `ES-ADIF-2027.pdf`, `ES-BOE-2024.pdf`
**IRG cross-check:** survey ES sheet consistent (canon A/B/C structure). ✓

### FI — Finland (Väylävirasto)

**Formula:** `TAC = gtkm × 0.002054`

**Values (1 Jan–31 Dec 2027):** basic component 0.2054 cents/gtkm. Additional charge for electric supply equipment 0.0167 cents/gtkm excluded (energy).

**Source:** Finnish NS 2027 (Table 2, ch. 5.3).
**On disk:** `FI-NS-2027.pdf`
**IRG cross-check:** survey consistent. ✓

### FR — France (SNCF Réseau)

**Formula:** `TAC = RC = gtkm × p_t + trkm × p_km`; **RM (market charge) for night trains = 0**

**Values (TT2027, non-contracted scale, App. 5.2.2, excl. VAT):**
- Conventional lines UIC 2–6: p_t = 5.705 EUR per 1,000 CGT-km (= 0.005705 EUR/gtkm), p_km = 0.657 EUR/trkm.
- Conventional lines UIC 7–9: 1.935 / 0.526.
- Night-train definition (segment): sleeper/couchette stock, > 5.5 h travel within at least 23:30–05:00, commercial paths from/to France; RM scale shows "–" for night trains → marginal-cost-only pricing (consistent with the Art.-34-type treatment previously identified).
- RCE 0.291 EUR/electric trkm excluded (energy). Flat 4.17 EUR/path-km for reservations not captured by IT systems — edge case, excluded.

**Assumptions:** UIC group 2–6 for main corridors (conservative; refine later via OSM `maxspeed`/`usage` proxy — UIC class is not in OSM).

**Source:** NS 2027 v3 (27 Apr 2026) §5.3; Appendix 5.2 "Scale of minimum services for the 2027 timetable" (11 Dec 2025).
**On disk:** `FR-DRR-2027-A52.pdf`, `FR-DRR-A512.pdf`
**IRG cross-check:** FR survey sheet structure matches (RC + RM per segment). ✓

### GR — Greece (OSE)

**Formula:** `C_MAP,p = infl × p_p × (c_T × d + c_wt × m × d [+ c_pss × stops])` → effective `trkm × 1.897 + gtkm × 0.004024`

**Values (NS 2026; 2019 base prices × inflation adjustment 1.1975; phased direct-cost recovery p_p = 0.60):** c_T = 2.64 EUR/km, c_wt = 0.00560 EUR/tkm → effective 1.897 EUR/trkm + 0.004024 EUR/gtkm. c_pss (4.21 EUR/stop, effective 3.02) deferred → §5. Electrification wear c_wte = 0.00210 EUR/tkm excluded (energy equipment).

**Assumptions:** the inflation multiplier is NS-2026-specific (cumulative CPI since 2019) and must be refreshed annually. Because that multiplier is the network statement's own step from the 2019 tariff to 2026 money, the **price basis recorded for GR is 2026, not 2019** — recording 2019 would let the general escalation re-apply seven years the source has already applied, overstating GR by about 23 %.

**Source:** OSE NS 2026 ch. 6.
**On disk:** `GR-OSE-2026.pdf`
**IRG cross-check:** survey EL sheet consistent. ✓

### HR — Croatia (HŽ Infrastruktura)

**Formula:** `TAC = T × Σ(Li × l) × Cvlkm` (passenger; excl. VAT)

**Values (TT 2026/27):** Cvlkm = 0.54 EUR/trkm (passenger); train-path equivalent T = 2.10 for EuroCity/**EuroNight**/InterCity; line parameter Li = 0.30–1.90 by category (L1 mainlines = 1.90). Tilting surcharge +0.20 (n/a). Electric-traction surcharge 0.10 EUR/trkm excluded (energy equipment). Ad-hoc +10–20 % (n/a for timetable paths).

**Assumptions:** L1 (1.90) on the international mainline corridors → 2.10 × 1.90 × 0.54 = 2.154 EUR/trkm; refine with Annex 5.1 line list when line-matching exists.

**Source:** HŽ NS 2027 §5.3 (items 4–18).
**On disk:** `HR-NS-2027.pdf`
**IRG cross-check:** survey HR sheet consistent (same formula). ✓

### HU — Hungary (MÁV; GYSEV not calibrated separately)

**Formula:** `TAC = trkm × (9 + rate(track cat)) + gtkm × 1.05` [HUF, excl. VAT]

**Values (TT 2026/27, MÁV summary table):** path ensuring 9 HUF/trkm (1 + 8 markup); passenger train-km part: cat I 1,134 / cat II 1,406 / cat III 1,204 HUF/trkm (charge + markup); gross-tonne-km part 1.05 HUF/gtkm. Catenary use 110 HUF/electric trkm excluded (energy). Station-use charges deferred → §5.

**Assumptions:** track-section category I for main international corridors → 1,143 HUF/trkm ≈ 2.9 EUR/trkm.

**Cross-check note:** ≈ 2.4× above IRG survey 2025 (1.18 EUR/trkm cat I) — consistent with the publicly protested Hungarian TAC increases from 2025/26 onward; table value confirmed directly in Annex 5.2-6, so retained.

**Source:** VPE/MÁV NS 2026–2027, Annex 5.2-6 (summary of network access charges).
**On disk:** `HU-NS-2627.pdf`

### IE — Ireland (Iarnród Éireann)

**Formula:** `TAC = trkm × 1.9268`

**Values (from Jan 2027):** variable usage charge 1.9268 EUR/trkm, single network-wide rate. Fixed track access charge applies only to franchised operators on ability-to-pay basis → assumed n/a for open access. DART traction-power charge excluded (energy).

**Source:** IÉ NS ch. 6.2/6.3.
**On disk:** `IE-NS-2027.pdf`
**IRG cross-check:** survey IE sheet empty; NS is primary. ✓ (one-sided)

### IT — Italy (RFI)

**Formula:** `TAC = gtkm × T_A1-2(speed class) + trkm × T_flat + trkm × CompB(segment, network, band)`

**Values (tariff year 2027, Listino PMdA 2025–2029):**
- Component A: T_A1-2 by speed class, e.g. [125–150) 0.001991, [150–175) ≤ 17 t/axle 0.002707 EUR/gtkm; T_flat = 0.185 EUR/trkm. TA3 contact-line component (0.241/0.482 EUR/trkm electric) excluded (energy equipment).
- Component B, segment **Basic**, band *Notturno*: 1.94–3.88 EUR/trkm depending on network part (LSE 2.54–2.75, FOND 2.20–2.43, COMPL 1.94–2.31, NODI 2.20–3.88).

**Mechanism & assumptions:** open-access night train = Basic segment (Servizio Universale contracted IC Notte would use the OSP-LP scale instead — switch by market segment input); **speed class for T_A1-2 taken from route average speed per leg** (model input) rather than a fixed class; Component B = FOND Standard as network pick, with the **Notturno band assumed 22:00–06:00** (`night_band` parameter; not confirmed in extracted excerpts) and applied per leg from the schedule — Diurno rates (Basic: 2.77–5.55 EUR/trkm) price the daylight fringes automatically.

**Source:** RFI Listino Tariffario PMdA (tariff period 2025–2029); RFI NS 2027.
**On disk:** `IT-LISTINO.pdf`, `IT-NS-2027.pdf`
**IRG cross-check:** survey IT sheet matches A+B structure. ✓

### LT — Lithuania (LTG Infra)

**Formula:** `TAC = gtkm × 0.0012`

**Values (TT 2026/27 tariff decision, 12 Dec 2025):** train traffic fee 0.0012 EUR/gross tkm; no markup for EU passenger services (transit-specific passenger tariff 0.0151 EUR/gtkm applies only to third-country transit). Contact-network fee 0.1709 EUR/trkm excluded (energy).

**Source:** LTG Infra tariff decision Nr. SPR-PAJ(INFRA)-138/2025; NS 2026–2027 §5.3.2.
**On disk:** `LT-LTG-2627.pdf`
**IRG cross-check:** survey LT sheet consistent. ✓

### LU — Luxembourg (ACF/CFL)

**Formula:** `TAC = trkm × c_C × α(bodies) × β(category) + trkm × c_A`

**Values (2026):** c_C = 2.771 EUR/trkm; towed passenger train > 8 bodies α = 1.1615; towed-passenger category β = 1.0355; path admin c_A = 0.05 EUR/km (regular timetable path). Scarcity charge 24.23 EUR/km applies only on declared saturated lines — currently none declared → 0. Electric supply c_E = 0.2583 EUR/trkm excluded (energy).

**Reference train:** ≈ 3.38 EUR/trkm.

**Source:** LU NS 2027 v1.0 §5.3.2.
**On disk:** `LU-NS-2027.pdf`
**IRG cross-check:** survey LU sheet consistent. ✓

### LV — Latvia (LDz / LatRailNet)

**Formula:** `TAC = trkm × 1.30 + gtkm × 0.00106848` (international passenger, wide gauge)

**Values (from 1 Jan 2026):** maintenance + traffic control 1.30 EUR/trkm (segment "international passenger services within EEA"); renewals 0.00106848 EUR/gtkm. Electric-traction supply-equipment 0.15 EUR/trkm excluded (energy).

**Source:** LDz NS 2027 §5.2 (LatRailNet board decisions 11–12/2025).
**On disk:** `LV-NS-2027.pdf`
**IRG cross-check:** survey LV sheet consistent. ✓

### NL — Netherlands (ProRail)

**Formula:** `TAC = trkm × rate(weight class)`

**Values (TT2027):** ≤120 t 0.5934; 121–160 t 0.7417; 161–320 t 0.9435; 321–600 t 1.3114; 601–3,200 t 2.2252; >3,200 t 2.7533 EUR/trkm. No markups on any segment (direct cost only — confirmed also in IRG survey: all markups 0). HRN levy applies to the domestic franchise only. Stop charges deferred → §5.

**Assumptions:** reference night train (≈ 640 t) → 2.2252; lighter compositions (< 600 t) drop to 1.3114 — model picks class from composition weight (available input, no assumption needed at runtime).

**Source:** ProRail NS 2027 v1.1 (train path service, section 4 user costs).
**On disk:** `NL-NS-2027.pdf`
**IRG cross-check:** survey 2025 class rates ≈ 15–18 % lower — plausible indexation; structure identical. ✓

### NO — Norway (Bane NOR)

**Formula:** `TAC = trkm × basic charge` (open-access commercial passenger pays **no markup**)

**Values (NS 2027, "2026 charges", indexed annually):** axle load < 25 t: Oslo region 5.84 NOK/trkm; Ofotbanen and remainder 9.94 NOK/trkm. Markups exist only for PSO, airport feeder, and ore segments → n/a for open-access night trains.

**Assumptions:** apply 9.94 NOK/trkm uniformly (conservative; Oslo-region km are cheaper).

**Source:** Bane NOR NS 2027 §5.3.3 (Table 4) + mark-up methodology report (Dec, NS25).
**On disk:** `NO-NS-2027.pdf`
**IRG cross-check:** survey NO sheet consistent. ✓

### PL — Poland (PKP PLK)

**Formula:** `TAC = trkm × 8.01 PLN × WM(mass) × WK(avg line category)`

**Values (price list effective 13 Dec 2026):** SMK = 8.01 PLN/trkm; WM: 600–660 t → 1.0000 (0.377–3.286 across 60–4,800 t); WK: avg category 2.1 → 1.0000 (0.672–1.179 across cat 4.0–1.0). Electric-traction component 0.29 PLN/km excluded (energy). Direct-cost-only (no markup in PLK basic fee).

**Assumptions:** WM = 1.0 (reference train), WK = 1.0 (avg category 2.1; mainlines trend better than average → mildly conservative) → 8.01 PLN/trkm ≈ 1.9 EUR.

**Source:** PLK NS 2026/2027 Annex 9.1 (Resolution 1048/2025, updated 14 Jan 2026).
**On disk:** `PL-PLK-A91.pdf`
**IRG cross-check:** survey PL sheet consistent (same SMK × WM × WK system). ✓

### PT — Portugal (Infraestruturas de Portugal)

**Formula:** `TAC = trkm × T(line cat, period, traction, segment)`

**Values (TT2027, excl. VAT), segment *International passenger*, electric:** Low band: cat A 2.16 / B 1.95 / C 1.84 EUR/trkm; Regular and Peak bands: A 2.55 / B 2.29 / C 2.16. Band hours (weekdays): Low 00:00–05:59 and 20:45–23:59, Regular 10:00–16:30, Peak 06:00–09:59 and 16:31–20:44; weekends/holidays: Low 00:00–05:59 and 20:45–23:59, Regular 06:00–20:44 (no peak).

**Mechanism & assumptions:** band selected **per leg from the actual schedule** (evening departures before 20:45 and morning arrivals after 06:00 price at Regular/Peak on those legs). Line category A assumed for the Norte/Sul mainlines. Diesel (NE) rates ~10 % lower if traction input says diesel.

**Source:** IP NS 2027 1st Addenda, §5.3 tariff table + line-category legend.
**On disk:** `PT-IP-2027.pdf`
**IRG cross-check:** survey PT sheet consistent. ✓

### RO — Romania (CFR SA)

**Formula:** `TAC = Σ_sections Km × ( Ttsn × [1 + (m − 60) × 0.00014] + Tc )` [lei]

**Values (valid from 1 Mar 2024; passenger):** Ttsn by line class A/B/C/D = 3.45 / 2.80 / 2.14 / 1.48 lei/trkm; Tc = 9.562 / 9.562 / 9.108 / 4.085 lei/trkm; Tmin = 60 t; Ft = 0.00014. Electrification Ttse 0.676 lei/trkm excluded (energy equipment). Verified against worked examples in Annex 26.a (e.g. class A, 500 t, non-electrified: 13.22 lei/trkm ✓). Commercial-stop charge deferred → §5. Rank-based IAC reductions (rank II–IV: 84–73 %) not applied (conservative).

**Assumptions:** line class A/B on main corridors → ≈ 13.2 lei/trkm at 400–600 t ≈ 2.6 EUR/trkm.

**Source:** CFR NS Annexes 25.a (methodology), 25.b (section classes), 26.a (tariff values + examples).
**On disk:** `RO-CFR-A25.pdf`
**IRG cross-check:** survey RO sheet consistent (same IAC structure). ✓

### SE — Sweden (Trafikverket)

**Formula:** `TAC = gtkm × track charge(mean axle load) + trkm × train path charge`

**Values (NS 2027 Annex 1B, edition 5 Dec 2025):** track charge passenger, mean axle load ≤ 17 t: 0.0218 SEK/gtkm (> 17 t: 0.0237); train path charge: 4.98 SEK/trkm (all segments). Öresund Link: passenger trains pay the regular track + train path charges only — the **passage charge (3,445.80 SEK) applies to freight exclusively** (→ §5a). Reference train: 600 × 0.0218 + 4.98 ≈ 18.1 SEK/trkm ≈ 1.6 EUR.

**Assumptions:** passenger coaches → ≤ 17 t mean axle load class (runtime check from composition axle data possible).

**Source:** Trafikverket NS 2027 §5.3 (Table 5.1) + Annex 1B "Charges for services" (pp. 212 ff.).
**On disk:** `SE-NS-2027.pdf`
**IRG cross-check:** survey 2025 (0.4366 EUR/trkm path charge ≈ 4.98 SEK; gtkm rates ≈ +10 % nominal) — consistent. ✓

### SI — Slovenia (SŽ-Infrastruktura)

**Formula:** `TAC = trkm × 2.01 × PP(line) × PD(len) × PM(wt) × PV(speed) × PTP × Pl(loco) − incentives`

**Values (TT2027):** C_P1 = 2.01 EUR; PP: R1 0.47 … R4 1.44 (all main corridors incl. Ljubljana–Sežana/Dobova/Jesenice/Šentilj and Koper line = R4); PD: > 300 m 1.05; PM: 251–1,000 t 0.75; PV: loco-hauled 0.97; PTP passenger 1.00; Pl: Vectron/Taurus-class e-loco 1.09. ETCS incentive −0.03 EUR/km if equipped.

**Assumptions:** reference train on R4 → 2.01 × 1.44 × (1.05 × 0.75 × 0.97) × 1.09 ≈ 2.41 EUR/trkm; no ETCS incentive claimed (conservative). Koper-line markup history noted but no passenger markup in the P1 formula.

**Source:** SŽ NS 2027 §5.3 (factor tables).
**On disk:** `SI-NS-2027.pdf`
**IRG cross-check:** survey SI sheet consistent (same factor system). ✓

### SK — Slovakia (ŽSR)

**Formula:** `TAC = trkm × (U1 + U2)(track cat) + gtkm/1000 × U3(track cat) × ke`

**Values (Measure 2/2018 Annex 1, unchanged since 1 Jan 2019, excl. VAT):**

| Track category | U1 (timetable train) €/trkm | U2 €/trkm | U3 €/1,000 gtkm |
|---|---|---|---|
| 1 | 0.0691 | 0.997 | 1.102 |
| 2 | 0.0566 | 0.927 | 1.048 |
| 3 | 0.0487 | 0.884 | 0.945 |
| 4 | 0.0319 | 0.774 | 0.779 |
| 5 | 0.0272 | 0.588 | 0.670 |

Ad-hoc U1 rates higher (0.0981–0.1890). ke = 1.2 for diesel traction on electrified lines, else 1.0. U4 (electric supply equipment, 0.228 EUR/1,000 gtkm) excluded (energy). Component check: U1+U2+U3+U4 at 1,000 t, cat 1, electric = 2.396 EUR/trkm = IRG survey composite. ✓

**Mechanism & assumptions:** weight applied at runtime via U3; track category 1–2 assumed for main corridors. Reference train (600 t, cat 1): 0.0691 + 0.997 + 0.6 × 1.102 = **1.73 EUR/trkm**.

**Source:** ŽSR NS 2027 Annex 5.2.B (Measure 2/2018 of the Transport Authority, Annex 1); NS 2027 ch. 5.3.
**On disk:** `SK-ZSR-A52B.pdf`

### UK — Great Britain (Network Rail)

**Formula:** `TAC = vehicle_miles × VUC(vehicle class)` → per train-km: `(VUC_loco + n_coach × VUC_coach) / 1.609`

**Values (CP7, 2023/24 prices, indexed within CP7):** default rates — locomotive 127.05 p/vehicle-mile, coach 23.45 p/vehicle-mile, MU motor 60.44, MU trailer 28.23. Reference train (1 loco + 10 coaches): 361.55 p/train-mile ≈ 2.25 GBP/train-km. Class-specific rates available in the price list (incl. Caledonian Sleeper Mk5 stock) for refinement. EAUC (electrification asset usage) excluded (energy equipment); fixed track access charges apply to franchised operators only → n/a open access.

**Channel Tunnel:** separate charging regime (Getlink, not Network Rail/CP7) → modelled as a crossing entity in the `passage_charges` table (§5a), now sourced from the Fixed Link Usage Annual Statement 2026. HS1 (London–tunnel) likewise has its own charging framework outside CP7 — flag if UK routing via HS1 enters the target network.

**Source:** Network Rail CP7 Track Usage Price List (xlsm), sheets "Passenger VUC" / "Default Passenger VUC"; NR NS 2027.
**On disk:** `UK-NR-CP7.xlsm`
**IRG cross-check:** survey UK sheet consistent (VUC per vehicle-mile system). ✓

---
"""

In [ ]:
SCOPE = r"""
## Scope

Model target: charge for the **minimum access package (MAP)** for a passenger night train, per routed leg.

Explicitly **excluded** (calculated separately later):

- Station charges and station-usage/stop charges — see §5 (deferred register) so nothing is lost or double-counted. Exception: the CH Haltezuschlag is a capacity element of the path price (no station-usage equivalent exists) and is included here.
- Energy-related charges: traction current, and all charges for the *use of electric supply equipment* (catenary/contact-line access, electrification wear, transformation/distribution canons). These are consistently excluded across countries even where they are formally part of the MAP, and are flagged per country.
- Shunting, parking/stabling, service facilities.
- Capacity-allocation admin fees where they are per-application flat fees (negligible); per-km reservation components are noted where material.
- Performance-regime payments (symmetric incentive schemes, net ≈ 0 in expectation).
- ~~Congestion/scarcity surcharges~~ — **no longer excluded.** The implemented model applies them as ACTIVE conservative defaults on the peak-overlapping share of a run (AT flat congestion surcharge 1.6081 EUR/trkm, CH peak factor 2 on the day rate; bands Mon–Fri 06:00–09:00 / 16:00–19:00, priced at 5/7 expected value since the model has clock minutes, not dates). Rationale: a night train's peak-hour approach into Wien Hbf or Zürich HB plausibly touches a declared high-load section, and treating every unconfirmed approach as free would systematically understate cost in exactly the pattern night trains run.
"""

SCHEMA = r"""
## Model inputs and component schema

Available inputs per routed leg: distance by country, average/max speed, travel time, schedule (calendar + clock time), stops, market segment, composition (weight, length, seats/berths, vehicles, axles, traction type).

Not reliably available: mapping to IM-catalogued line categories (UIC class, national price-list categories). Where charges depend on line category, a **conservative fixed assumption** is made (documented per country) — conservative = do not underestimate the charge; main international corridors are assumed to be in the higher-priced mainline categories. OSM proxies (`maxspeed`, `highspeed`, `usage`) can refine this later.

Per-country calibrated columns:

| Column | Unit | Meaning |
|---|---|---|
| `per_train_km` | national currency / train-km | All train-km-proportional components incl. applicable markups |
| `per_gtkm` | national currency / gross-tonne-km | Weight-proportional component |
| `per_seat_km` | national currency / seat-km | Capacity-proportional component (CH fallback mode) |
| `per_stop` | national currency / stop | Capacity-type stop surcharges that are part of the track charge (CH Haltezuschlag only; all station-usage stop fees stay deferred, §5) |
| `revenue_share` | fraction of traffic revenues | Revenue-based contribution margin (CH primary mode) |
| `fixed_per_crossing` | national currency / train | Crossing-specific passage charges (§5a) |
| multipliers | — | Documented factors already folded into the values above |

Values are **calibrated** in national currency at the documented price
basis, because that is what the source document says and provenance is
checked against the source. They are **stored and consumed in EUR**:
`06_seed_export.ipynb` converts each component once against its own
sourced currency (`FX_TO_EUR`) and, per the escalation section below,
carries it from its basis year to the evaluation year. The database
holds one plain EUR number per component; `calc_tac.py` never sees a
native currency or a price basis. Both the summary table and the
country sections below show native and EUR side by side.

**Runtime mechanisms** (no fixed assumptions needed where the model has the input):

1. **Time-dependent rates:** each leg carries start/end clock time, so time-banded tariffs are applied per leg via `rate(band(t))`, not via a blanket "night trains run off-peak" assumption. Applies to: CH peak factor (legs touching Mon–Fri 06:00–09:00 / 16:00–19:00 on listed high-load sections — relevant for arrivals after 06:00), BE period classes, PT schedule bands, IT Notturno/Diurno bands, DE Nacht pro-rata rule, FR night-train qualification. Band definitions are given per country below.
2. **Weight-dependent rates:** exact composition gross weight is a model input; weight-class lookups (NL, PL WM, LU α, SI PM) and weight formulas (CZ, RO, AT/FR/FI/… gtkm terms) are evaluated at runtime. Reference-train figures below are indicative only.
3. **Speed-class rates (IT):** speed class taken from route average speed per leg (model input).
4. **High-speed vs conventional line (ES, FR):** selected per leg from infrastructure max speed (OSM `maxspeed`/`highspeed`); e.g. ES line type A when max speed ≥ 250 km/h.
5. **Crossing-specific passage charges:** modelled as a separate `passage_charges` table keyed by crossing entity (Storebælt, Øresund, Channel Tunnel, …), triggered by route geometry — independent of the per-country km-based charges. See §5a.

**Reference train** for the indicative summary: 1 electric locomotive + 10 coaches, 600 t gross, ~300 m, 500 places, timetable path, running in the night band.
"""

DESIGN = r"""
# Part II — Calculation design

What `calc_tac.py` implements, and why. These decisions were taken
2026-07-28 while turning the tariff facts of Part I into running code;
they are as much in need of review as the numbers, since several encode
judgement calls that a reader should be able to disagree with.

### Two night mechanisms, not three

`track_tac_night_mode` has exactly two values: `none` and `time_band`. An
earlier `segment` mode for Germany was dropped — the DE tariff is a band
tariff (Nacht 23:00–06:00, pro-rata like IT/BE/PT), and the SPFV rule is
a band *widening*, not a separate mechanism: a train carrying night
accommodation is priced Nacht over its **entire German run**. That
widening is the boolean `track_tac_night_full_if_accommodation`, and it
is evaluated **timetable-independent from the composition**
(`Composition.has_night_accommodation`: any place whose `class_main` is
not Seat or Catering — a dining car alone does not make a night train).
CH deliberately sits in `none`: its 22:00–06:00 band belongs to
electricity pricing (03), not track access.

### NULL means "not levied" — group resolution only

A NULL component is a documented tariff fact (FI charges per
gross-tonne-km *only*; its NULL `b_day` must price at zero), never
missing data. Per-field default substitution would therefore corrupt
calibrated countries. Resolution against the default row happens as a
**group**, only when `b_day`, `b_night` and `gamma` are ALL NULL
(`DBDataLoader._row_to_track`) — i.e. the country has no usable charging
term at all (in practice only CY/MT, which have no railway). The default
group is `b_day` only, set to the median of the calibrated per-country
NT-REF rates: a synthetic country charging both a train-km AND a tonne-km
term would overstate.

### The flat column is display-only

`track_tac_eur_train_km` stays in the schema as the indicative NT-REF
rate for tables and the frontend, but **the cost model never reads it**.
`tests/test_72_calc_tac_units.py` enforces this by construction: every
track stub carries an absurd flat (999 999) and every assertion still
reproduces the calibration numbers. SE keeps its flat NULL in the seed —
it is the suite's is_default fixture (test_04) and loses nothing, since
its component group is seeded normally.

### Clock placement inside a segment

A segment's country windows are placed on the clock by walking
`segment.countries` (the ordered path from routing, added by the
ROUTE_BUILDER bump this work carries)
and splitting the segment's departure→arrival span by
`country_time_shares`. Band overlaps are then computed per window with
wrap-safe minute arithmetic (`models/utils.band_overlap_min`), never by
picking the window's midpoint. `UNK` slices (ferries, open water) own
time but no tariff: they advance the cursor without being charged, so the
country after a crossing lands at the correct clock time. Route payloads
predating that bump carry no `countries` list; the serializer falls back to
the share dict's keys — segment stays evaluable, path ordering (and with
it exact clock placement for multi-country segments) is approximated.

### Peak surcharges: active conservative defaults

AT's congestion surcharge (flat EUR/train-km) and CH's peak factor (×2 on
the day-rate term) apply to the peak-overlapping share of a run. Both
formally apply only on declared overloaded/high-load sections; the model
applies them blanket on the peak overlap because a night train's morning
approach into Wien Hbf or Zürich HB plausibly touches such a section, and
pricing every unconfirmed approach as free would systematically
understate cost in exactly the pattern night trains run. Weekday-only
bands are priced at their expected value, `WEEKDAY_BLEND = 5/7` — the
model has clock minutes, not service dates. The two terms are kept as
separate result fields (`congestion_eur` folded next to the multiplier's
base effect) so views can show a congestion charge as such.

### Per-stop terms

The CH Haltezuschlag is a capacity element of the path price, not a
station-usage fee — it belongs to TAC (no double-counting with the stop
charge model). Each stop is charged at **its own country's** per-stop
rate; a trip's first segment charges both of its ends (the origin stop
would otherwise never be counted).

### Passages

Storebælt, Øresund and the Channel Tunnel are separately charged
crossings — per traverse, not per km — modelled as dedicated entities in
`input_params.passage_charges` (the fifth scenario-versioned table, same
full-snapshot contract). Detection happens at ROUTING time
(`rail_router.PassageIndex`): the first trip leg intersecting a crossing
polygon owns it, recorded on the segment as `passages`. Øresund is one
polygon with two charge rows (`OERESUND_DK` / `OERESUND_SE`) — each IM
bills its half of the crossing. The Channel Tunnel's per-passenger term
is evaluated against the segment's passenger load per train run (annual
demand ÷ operating days, distributed along the OD path in the traffic
pre-pass). An unknown passage id in a payload is warned and skipped, not
fatal — a stale route must stay evaluable.

### Revenue share (CH Deckungsbeitrag)

First-class parameter, fully plumbed (`revenue_share × attributable
segment revenue`), currently NULL everywhere because the authority-set
percentage is not published. The traffic pre-pass in
`models/evaluation/calc.py` distributes route revenue distance-weighted
along each OD path, so the moment a value lands in the DB the charge
computes without code changes.

### Currency and price basis

All DB values are EUR. FX conversion happens exactly once, in
`06_seed_export.ipynb` (`FX_TO_EUR`), per component using each sourced
value's own currency. `calc_tac.py` never sees native currency.

### Locomotive weight

Tonnage terms price the full consist: coaches plus
`n_locos × loco_weight_t`. 90 t (Vectron-class) is a constant on every
catalog composition until the compositions calibration workbook carries a
per-type value (`composition_type_loco_weight_t`).

### Known understatements (accepted, documented)

- **BE day fringe**: only `b_night` is calibrated; the fraction of a
  Belgian run outside the 19:00–05:59 band prices at zero instead of the
  (higher) day/peak coefficients. Understates BE for early-evening
  departures.
- **CY/MT**: no railway, no seed row — their TAC columns are NULL and the
  loader substitutes the default group with a warning. Unreachable by
  routing, so this never prices a real leg.
"""

ESCALATION = r"""
## Price basis and escalation to 2032

**The problem.** Calibrated TAC values sit at the price basis of the
document they came from — 22 values at 2027, 15 at 2026, and a tail at
2025, 2024, 2023 and 2019 (GR, whose rates are a 2019 base the NS itself
inflates). The evaluation year is **2032**. Nothing currently bridges
that gap, so every charge is priced at its document year and the model
silently assumes track access is flat for five to thirteen years.

**Why that is not a defensible default.** The compositions calibration
already escalates: recurring operating rates are carried to *nominal
2032* (wage chain for labour, 2 %/yr general otherwise), and external
benchmarks are normalised to 2032 before comparison, on the stated
grounds that "a 2012-price corridor against 2032 rates is an optical
illusion, not a validation". Leaving TAC at 2026/27 while operator cost
sits at 2032 mixes price bases **inside one cost stack** — it does not
merely add uncertainty, it biases the infrastructure share of total cost
downwards, which is exactly the number this project exists to argue
about.

**What the evidence says.** Passenger MAP charges per train-km, IRG-Rail
European average:

| Period | Evolution | Source |
|---|---|---|
| 2015 → 2019 | EUR 4.13 → 4.63, **+2.9 %/yr** | IRG-Rail 9th MM Report |
| 2020 → 2024 | **+3 %/yr**, risen continuously | IRG-Rail 14th MM Report |
| DE SPFV 2020 → 2024 | +19.9 % total, **+4.6 %/yr** | BNetzA via Wirtschaftsdienst 2026 |

Two readings matter. First, the European passenger average is
**remarkably stable at ~3 %/yr across a decade** spanning both the
low-inflation 2010s and the 2022–23 inflation spike — it is not an
artefact of one period. Second, that ~3 % sits *above* general HICP
inflation (~2 % target), i.e. track access has risen ~1 %/yr in real
terms. The mechanism is structural rather than cyclical: full-cost
recovery under Directive 2012/34 Art. 31–32 combined with a growing
maintenance and renewal burden, so the charge base rises faster than
consumer prices.

Germany runs hotter than the European average, but for a reason that is
now ending rather than compounding: the *Trassenpreisbremse*
(ERegG §37(2)) capped SPNV increases, which forced the shortfall onto
SPFV and freight. The ECJ struck that provision down in March 2026
(C-770/24) as incompatible with EU law. The German SPFV escalation of the
last five years therefore should **not** be extrapolated — the
cross-subsidy driving it is being unwound, and the direction of the
correction is downward for SPFV.

**Recommendation: one documented rate, 3 %/yr, applied per value from its
own basis year.**

> `value_2032 = value_basis × (1 + tac_escalation_per_year) ^ (2032 − basis_year)`

with `tac_escalation_per_year = 0.03`. `basis_year` is already a column
in `tac_components.csv`, so no new calibration input is needed. Resulting
uplifts:

Countries carrying a national deviation are excluded from this table —
listing them here would imply they move with the average.

| Basis year | Values | Years to @@TARGET_YEAR@@ | ×@@ESC_LOW@@ | ×2.5 % | **×@@ESC_RATE@@** | ×@@ESC_HIGH@@ |
|---|---|---|---|---|---|---|
@@ESCALATION_TABLE_ROWS@@

Weighted across all @@N_ESCALATED@@ escalatable values including the
deviations, this is a **+@@MEAN_UPLIFT@@ %** uplift on the infrastructure
charge line.

**Why one European rate rather than per-country rates.** Per-country
escalation would be more precise in principle and less honest in
practice: it would need a defensible forward rate for 28 regulatory
regimes, and the only forward-looking evidence available is national
regulatory intent, which is neither uniform nor reliably published five
years out. A single documented rate with a stated band is easier to
challenge and easier to move.

**National deviations.** An average is the wrong instrument where a
national tariff is demonstrably not moving with it. Those cases take an
explicit rate of their own, stated here rather than buried in a note,
because a deviation is a claim about one country's next decade and
should be as challengeable as the average it replaces. @@N_OVERRIDE_VALUES@@
of the @@N_ESCALATED@@ escalatable values are affected:

| Country | Applied rate | Reason |
|---|---|---|
@@OVERRIDE_ROWS@@

A *known* structural change is a different thing again and belongs in the
country's own note — the AT market-markup reintroduction planned for 2029
(+1.12 EUR/trkm commercial passenger) is exactly such a case, falls
inside the @@TARGET_YEAR@@ horizon, and is flagged in the AT section.

**Uncertainty band:** 2 %/yr (pure HICP, assumes the real-terms rise
stops) to 3.5 %/yr (the 2015–2024 trend continues and steepens with the
renewal backlog). Carry as `low`/`high` so it reaches the sensitivity
table — at ±0.5 pp the spread on the 2027 cohort is roughly ∓2.5 pp of
the uplift, which is small next to the ±50 % class uncertainties
elsewhere in the model, but it is the difference between an escalation
that is defended and one that is assumed.

**Implementation:** the factor belongs in `06_seed_export.ipynb`,
applied per value alongside the existing `FX_TO_EUR` conversion, so the
DB continues to hold one plain EUR number per component and
`calc_tac.py` stays unaware of both currency and price basis. It is a
scenario parameter, not a constant: a 2032 scenario and a 2040 scenario
should differ by their target year, not by re-calibration.

**Open question for the review:** the same argument applies to the
electricity (§2), facility (§3) and route-context (§4) domains, whose
values also sit at document-year basis. Whatever rate is agreed here
should be applied consistently across all four, or the mixing problem
simply moves rather than resolves.
"""

PASSAGES = r"""
## Crossing-specific passage charges (`passage_charges` table)

Fixed per-train charges tied to specific crossings, triggered by route geometry, additive to the per-country km-based charges:

| Crossing | Charged by | Passenger charge per train | Status |
|---|---|---|---|
| Storebælt | Banedanmark | 4,876.73 DKK (2025 order, excl. VAT) | sourced |
| Øresund — Danish part | Banedanmark | 2,592.02 DKK (2025 order, excl. VAT) | sourced |
| Øresund — Swedish part | Trafikverket | **none for passenger** (freight only: 3,445.80 SEK); regular SE track/path charges apply on the link | sourced (NS 2027 Annex 1B) |
| Channel Tunnel | Getlink (Eurotunnel) | Offer 1 (regular weekly paths), **night trains @120 km/h, off-peak**: reservation fee ≈ 4,039 EUR/train o/w (2,255 € + £1,486) **+ per-passenger access fee ≈ 18.35 EUR/pax o/w** (8.36 € + £8.32); maintenance periods @100 km/h: ≈ 6,732 EUR/train. 2020 price basis, indexed (RPI/IPC; pax fee −1.1 % p.a. factor); billed half EUR / half GBP (combined at £1 = 1.20 €) | sourced (Fixed Link Usage Annual Statement 2026, Annexe 4) |

Architecture note: keep these as a dedicated `passage_charges` entity keyed by crossing (extensible for Fehmarnbelt in the 2032 scenario), not as country attributes. The entity needs `fixed_per_train` **and** `per_passenger` columns — the Channel Tunnel toll has a per-carried-passenger component, which couples the passage cost to the demand model output (load-dependent, evaluated in the revenue/cost loop, not the routing stage). Getlink specifics: the Fixed Link NS states night passenger trains operate at 120 km/h in off-peak periods (100 km/h in maintenance periods) → the off-peak row is the night-train tariff by definition; Offer 2 (individual trains) runs ~10 % higher plus 7,500 EUR admin per contract — Offer 1 (regular weekly paths) is the correct basis for a scheduled night service.
"""

DEFERRED = r"""
## Deferred stop-based components (→ station-charge calibration)

Excluded from this model to avoid double counting with the later station model; values recorded so they are not lost. The CH Haltezuschlag was moved **into** the TAC model (capacity price element of the path price, no CH station charge exists for platform provision — see CH section).

| Country | Component | Value | Nature |
|---|---|---|---|
| CZ | C_PK passenger platform access | 0.04–0.11 CZK per stop·t by station category (2027) | Platform access within statutory two-component price |
| GR | c_pss | 4.21 EUR/stop × 0.7185 ≈ 3.02 EUR/stop | Stop term of the C_MAP formula |
| NL | Stop charge by station type | 0.09 / 0.36 / 0.88 EUR/stop (2025 survey values; 2027 values in NS stop-service table) | MAP category-1 stop service |
| HU | Use of stations by passenger trains | 3,327–3,877 HUF/stop; origin/destination 3,345–3,669 HUF | Station usage (IM-levied) |
| RO | Commercial stop charge (Annex 26.a §2.1) | not extracted | Station stop charge |
"""

ACTIONS = r"""
## Open actions carried into the review

1. **DE −17 %** — applied as assumption (calibrated 2.76 EUR/trkm); re-approval document itself still to be filed. → confirm and archive source.
2. **CH revenue_share** — authority-set contribution-margin percentage for international night trains not published in the legal texts (observed range 0–21 %). → scenario parameter; check BAV/concession publications.
3. **IT Notturno band hours** — assumed 22:00–06:00 pending confirmation from RFI PIR.
4. **Channel Tunnel price basis** — Getlink scales are at 2020 prices with monthly/annual indexation (RPI/IPC, pax fee −1.1 % p.a.); indexation to the model's price-basis year still to be applied. HS1 charging framework not sourced (only relevant if London routing enters the target network).
5. **EE Edelaraudtee** and **HU GYSEV** sub-networks not separately calibrated (minor route shares).
6. Line-category matching (price-list categories PL/RO/SK/HU/PT/SI/HR/CH, UIC classes FR) fixed by conservative assumption; ES/FR high-speed vs conventional now resolved at runtime from infra max speed; OSM proxy (`maxspeed`/`highspeed`/`usage`) is the refinement path for the rest — confirmed approach.
"""

In [ ]:
CALIBRATION_TEMPLATE = r"""# Track Access Charges — Calibration

Charges for the **minimum access package (MAP)** on a passenger night
train, per routed leg, for @@N_COUNTRIES@@ European countries.

Generated by `02_tac_calibration.ipynb` on @@GENDATE@@ — do not edit by
hand; re-run the notebooks (01 then 02, top to bottom) to regenerate this
document and the CSVs under `data/`.

Scope is deliberately narrow. Energy, station and facility charges are
calibrated in their own domains and are excluded here even where a
national tariff bundles them; the scope section says exactly what is left
out and why. Implementation:
`backend/models/infrastructure/calc_tac.py`.

**Provenance at a glance:** @@N_SOURCED@@ values read directly from a
named locator, @@N_DERIVED@@ derived by documented arithmetic,
@@N_ASSUMED@@ assumed with a band, across @@N_CITED@@ cited sources.
Everything else is `missing` or `no_railway` — explicit rather than
absent, so a gap cannot be mistaken for a zero.

---

# Part I — Scope and schema

@@SCOPE@@

@@SCHEMA@@

---

@@DESIGN@@

---

# Part III — Per-country calibration

## Summary table

Every rate as published in its **native currency**, converted to **EUR**,
with the price basis it carries. The final column prices the reference
train — 1 locomotive + 10 coaches, @@TARGET_YEAR@@ money, 600 t gross,
~300 m, 500 places, running in the night band — at the rate term plus the
gross-tonne-km term, **after** escalation to @@TARGET_YEAR@@.

| Country | Rate term | Native | → EUR | γ native | → EUR | Basis | **@@TARGET_YEAR@@ EUR/train-km @600 t** |
|---|---|---|---|---|---|---|---|
@@SUMMARY_TABLE_ROWS@@

CY and MT have no railway network — a positive fact, not an uncalibrated
country. Non-EU transit states (RS, MK, XK) are out of scope for this
round; IRG survey sheets exist if they enter the target network.

## Currency conversion

@@N_FX_COUNTRIES@@ countries publish in a currency other than the euro
(@@FX_CURRENCIES@@), so the FX table is a calibration input in its own
right rather than a formatting detail — it is pinned, dated and cited
like any other value. ECB reference rates, snapshot @@FX_SNAPSHOT@@:

| Currency | Reference rate | → EUR factor |
|---|---|---|
@@FX_TABLE_ROWS@@

Conversion happens **exactly once**, in this notebook, per value against
its own sourced currency. The database holds one plain EUR number per
component and `calc_tac.py` never sees a native currency. A scenario may
legitimately pin a different snapshot; nothing downstream needs to change.

@@ESCALATION@@

---

@@COUNTRY_SECTIONS@@

---

@@PASSAGES@@

@@DEFERRED@@

@@ACTIONS@@

## Sources

Documents a calibrated value leans on directly. Every one is checked
against the register written by `01_source_extraction.ipynb`, so a
citation that does not resolve fails the notebook rather than reaching
the document.

| source_id | Document | Publisher | Status | Link |
|---|---|---|---|---|
@@SOURCE_TABLE_ROWS@@

Cross-check, conversion and method sources — these do not price a
specific country, they underpin the FX table, the escalation rate and the
scope decisions:

| source_id | Document | Publisher | Link |
|---|---|---|---|
@@METHOD_SOURCE_ROWS@@

Stored documents follow `{source_id}.{ext}`, so the filename is derivable
from the register and needs no column of its own.
"""

In [ ]:
# --- per-country EUR lines, injected into the prose blocks ---
_LABEL = {
    "b_day": "day/base rate",
    "b_night": "night rate",
    "gamma": "per gross-tonne-km",
    "seat_km": "per seat-km",
    "per_stop": "per stop",
    "fixed_per_train_km": "fixed per train-km",
    "congestion_surcharge_eur_km": "congestion surcharge",
}


def _eur_line(cc: str) -> str:
    """Native → EUR → evaluation year, for one country. Emitted for every
    country: euro countries still need the escalation shown."""
    parts, currencies = [], set()
    for p in PARAMETERS:
        v = _get(cc, p)
        if v is None or v.unit == "factor":
            continue
        currencies.add(v.currency)
        native = f"{_fmt_native(v.value, v.currency)} {v.currency}"
        eur = _fmt_eur(v.eur)
        model = _fmt_eur(v.model_value)
        if v.currency == "EUR":
            parts.append(
                f"{_LABEL.get(p, p)} {native} → **{model}** ({v.basis_year}→{TARGET_YEAR})"
            )
        else:
            parts.append(
                f"{_LABEL.get(p, p)} {native} = {eur} EUR → **{model}** "
                f"({v.basis_year}→{TARGET_YEAR})"
            )
    if not parts:
        return ""
    fx_note = "" if currencies == {"EUR"} else f"FX at the {FX_SNAPSHOT} snapshot; "
    rate = escalation_rate(cc)
    esc_note = (
        f"escalated at {rate:.0%}/yr"
        if cc not in ESCALATION_OVERRIDE
        else f"**not escalated** ({rate:.0%}/yr, national deviation — see below)"
    )
    return f"**Seeded value** ({fx_note}{esc_note}): " + "; ".join(parts) + "."


def _override_note(cc: str) -> str:
    if cc not in ESCALATION_OVERRIDE:
        return ""
    rate, reason = ESCALATION_OVERRIDE[cc]
    return (
        f"**Escalation deviation ({rate:.0%}/yr instead of "
        f"{TAC_ESCALATION_PER_YEAR:.0%}):** {reason}"
    )


_country_blocks = []
for _block in COUNTRY_PROSE.split("\n### "):
    if not _block.strip():
        continue
    _block = _block if _block.startswith("### ") else "### " + _block
    _cc = _block[4:6]
    _line = _eur_line(_cc)
    _dev = _override_note(_cc)
    if _dev:
        _line = _line + "\n\n" + _dev
    if _line:
        _block = _block.replace("**Source:**", _line + "\n\n**Source:**", 1)
    _country_blocks.append(_block.rstrip())
COUNTRY_SECTIONS = "\n\n".join(_country_blocks)

# --- sources actually cited, read back from the register ---
with open(DATA_DIR / "sources_register.csv", encoding="utf-8") as fh:
    _register = {r["source_id"]: r for r in csv.DictReader(fh)}

_cited = sorted(
    {v.source_id for v in values if v.source_id}
    | {r["source_id"] for r in passage_rows if r["source_id"]}
)
SOURCE_TABLE_ROWS = "\n".join(
    f"| `{sid}` | {_register[sid]['title']} | {_register[sid]['publisher']} | "
    f"{'on disk' if _register[sid]['downloaded'] == 'x' else 'link only'} | "
    f"{_register[sid]['url_or_file']} |"
    for sid in _cited
)
_method_sources = [s for s in _register if s not in _cited]
METHOD_SOURCE_ROWS = "\n".join(
    f"| `{sid}` | {_register[sid]['title']} | {_register[sid]['publisher']} | "
    f"{_register[sid]['url_or_file']} |"
    for sid in _method_sources
)

TOKENS = {
    "@@SUMMARY_TABLE_ROWS@@": SUMMARY_TABLE_ROWS,
    "@@ESCALATION_TABLE_ROWS@@": ESCALATION_TABLE_ROWS,
    "@@OVERRIDE_ROWS@@": OVERRIDE_ROWS,
    "@@N_OVERRIDE_VALUES@@": str(_n_override_values),
    "@@FX_TABLE_ROWS@@": FX_TABLE_ROWS,
    "@@COUNTRY_SECTIONS@@": COUNTRY_SECTIONS,
    "@@SOURCE_TABLE_ROWS@@": SOURCE_TABLE_ROWS,
    "@@METHOD_SOURCE_ROWS@@": METHOD_SOURCE_ROWS,
    "@@SCOPE@@": SCOPE,
    "@@SCHEMA@@": SCHEMA,
    "@@DESIGN@@": DESIGN,
    "@@ESCALATION@@": ESCALATION,
    "@@PASSAGES@@": PASSAGES,
    "@@DEFERRED@@": DEFERRED,
    "@@ACTIONS@@": ACTIONS,
    "@@TARGET_YEAR@@": str(TARGET_YEAR),
    "@@ESC_RATE@@": f"{TAC_ESCALATION_PER_YEAR:.0%}",
    "@@ESC_LOW@@": f"{TAC_ESCALATION_LOW:.1%}",
    "@@ESC_HIGH@@": f"{TAC_ESCALATION_HIGH:.1%}",
    "@@MEAN_UPLIFT@@": f"{100 * (_mean_uplift - 1):.0f}",
    "@@N_ESCALATED@@": str(_n_esc),
    "@@FX_SNAPSHOT@@": FX_SNAPSHOT,
    "@@FX_CURRENCIES@@": ", ".join(_fx_used),
    "@@N_FX_COUNTRIES@@": str(
        len({v.country_code for v in values if v.currency != "EUR"})
    ),
    "@@N_SOURCED@@": str(sum(1 for v in values if v.status == SOURCED)),
    "@@N_DERIVED@@": str(sum(1 for v in values if v.status == DERIVED)),
    "@@N_ASSUMED@@": str(sum(1 for v in values if v.status == ASSUMED)),
    "@@N_COUNTRIES@@": str(len({v.country_code for v in values})),
    "@@N_CITED@@": str(len(_cited)),
    "@@GENDATE@@": CALIBRATION_REVIEWED,
}

import re as _re

# Prose blocks carry tokens of their own (the escalation section embeds the
# deviation table), so substitution runs to a fixed point rather than once.
doc = CALIBRATION_TEMPLATE
for _pass in range(5):
    _before = doc
    for _tok, _val in TOKENS.items():
        doc = doc.replace(_tok, _val)
    if doc == _before:
        break
else:
    raise RuntimeError("token substitution did not converge — cyclic token?")

_left = sorted(set(_re.findall(r"@@[A-Z0-9_]+@@", doc)))
assert not _left, f"unsubstituted tokens: {_left}"

_doc_path = DATA_DIR.parent / "TAC_CALIBRATION.md"
with open(_doc_path, "w", encoding="utf-8", newline="\n") as fh:
    fh.write(doc)
print(
    f"TAC_CALIBRATION.md: {len(doc.splitlines())} lines, "
    f"{len({v.country_code for v in values})} countries, "
    f"{len(_cited)} sources cited"
)